<p align='center'><img src="SP images/voting_stickers.jpg" width = 100 />&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;<img src="SP images/voting_stickers.jpg" width = 100 />

<!-- source: https://worksheets.clipart-library.com/worksheet/printable-voting-stickers-5.html -->

# <p align='center'>Voting Systems Introduction

## Open in Google Colab

[Open this notebook in Google Colab](https://colab.research.google.com/github/gmorre/Voting-Systems-alpha/blob/main/Voting_Systems_alpha.ipynb). Select **Connect**, then choose **Runtime → Run all**. The first code cell sets up the notebook automatically. Some activities ask you to enter a response. To keep your own changes, use **File → Save a copy in Drive**; that will not change the original on GitHub.

## What are voting systems?

<font size="4">Voting systems are algorithms that take the preferences of a group of individuals and converts those preferences into a choice for the entire group. In other words, a voting system determines the rules used to conduct an election. Mathematically speaking they are social choice functions. The input value is the set of invidual preferences, the output is the group choice. Social choice research spans many different fields: mathematics, economics, data science, political science and more.

Voting systems are an integral part of every democratic society. National, regional and local governments are chosen using these systems. They are also used by non-governmental organizations, professional associations, and corporations. They are even used by sports leagues and the entertainment industry for awards. These systems have a huge impact on the lives of everyone!

Is there a voting system that is better than all of the others? If the election has only two candidates contesting, then the answer is yes. If there are three or more candidates contesting the election, the answer is more complicated (and interesting!) 

Note that all of the election methods discussed in this notebook are for single-winner elections. These elections produce a single outcome, not a set of outcomes. In other words, only one candidate is elected.

## In this Jupyter Notebook you will learn 

<font size="4">

- how different voting systems work,
- pros and cons of the voting systems
- the fairness criteria and defects of voting systems,
- and the consequences of Arrow's Impossibility theorem.


## How to use this notebook:

<font size="4">

- run Python cells where indicated: select cell and then press shift + enter/return, in some cases you will be prompted to enter input
- avoid typing in Python cells
- read the information in each section
- have some paper and a writing utensil handy
- read examples, work through some parts where indicated
- do the activities!

## VoteKit

<font size="4">VoteKit is a Python package developed by the Metric Geometry and Gerrymandering Group (MGGG) that supports computational social choice research. This package was essential in creating this notebook. Many thanks to MGGG and all those who contributed to it. 

## Set up the notebook

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/gmorre/Voting-Systems-alpha.git"

if Path("/content").is_dir():
    repository_dir = Path("/content/Voting-Systems-alpha")
    if not repository_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(repository_dir)],
            check=True,
        )
else:
    repository_dir = Path.cwd()

if not (repository_dir / "requirements.txt").is_file():
    raise FileNotFoundError(
        "Open this notebook from the Voting-Systems-alpha repository folder."
    )

%cd {repository_dir}
%pip install -r requirements.txt

## Load the notebook packages

In [ ]:
from votekit.ballot import Ballot
from votekit.pref_profile import PreferenceProfile
from votekit.pref_profile import profile_df_head
from votekit.elections import Plurality 
from votekit.elections import IRV
from votekit.elections import Borda 
import numpy as np
import pandas as pd
from turtle import pd
from votekit.cleaning import remove_and_condense_rank_profile
import votekit.ballot_generator as bg
from votekit import PreferenceInterval
import csv
from votekit.cleaning import remove_cand_rank_profile, condense_rank_profile, remove_repeat_cands_rank_profile
from votekit.graphs import PairwiseComparisonGraph
import importlib
import voting_functions
from votekit.elections import Approval
from votekit.utils import score_profile_from_ballot_scores
from votekit.pref_profile.utils import convert_rank_profile_to_score_profile_via_score_vector
import poliscipy
from poliscipy.shapefile_utils import load_shapefile
from poliscipy.plot import plot_electoral_map

importlib.reload(voting_functions)
from voting_functions import *

# Section 1: Ranked Ballots

<font size="4">A ranked ballot is a ballot in which the voter ranks the choices in order of preference. Ranking 1 is often referred to as first-choice, ranking 2 second-choice and so on.

<font size="4">Ranked ballots are not the typical ballots used in the United States though they are becoming more common. We need to learn about ranked ballots because they are necessary for several of the voting systems you will learn about in this notebook.

## Example 1.1
<font size="4">Suppose 21 voters are given choices A, B and C. Run the following cell to view the ballots of the first seven voters. Each column shows the ballot of a single voter. (The Ballot index 0 column represents the ballot of the first voter.) Then run the next two cells to view the rest of the ballots.</font>

In [ ]:
ballots=[Ballot(ranking=[{"A"},{"C"},{"B"}]),Ballot(ranking=[{"A"},{"B"},{"C"}]),Ballot(ranking=[{"B"},{"C"},{"A"}]),Ballot(ranking=[{"C"},{"A"},{"B"}])
,Ballot(ranking=[{"C"},{"A"},{"B"}]),Ballot(ranking=[{"A"},{"C"},{"B"}]),Ballot(ranking=[{"C"},{"B"},{"A"}]),Ballot(ranking=[{"B"},{"C"},{"A"}])
,Ballot(ranking=[{"C"},{"A"},{"B"}]),Ballot(ranking=[{"B"},{"A"},{"C"}]),Ballot(ranking=[{"A"},{"C"},{"B"}]),Ballot(ranking=[{"B"},{"C"},{"A"}])
,Ballot(ranking=[{"A"},{"B"},{"C"}]),Ballot(ranking=[{"C"},{"B"},{"A"}]),Ballot(ranking=[{"C"},{"B"},{"A"}]),Ballot(ranking=[{"B"},{"A"},{"C"}])
,Ballot(ranking=[{"C"},{"B"},{"A"}]),Ballot(ranking=[{"A"},{"B"},{"C"}]),Ballot(ranking=[{"A"},{"B"},{"C"}]),Ballot(ranking=[{"B"},{"C"},{"A"}])
,Ballot(ranking=[{"A"},{"B"},{"C"}])]
candidates=["A","B","C"]
profile = PreferenceProfile(ballots=ballots, candidates=candidates)

profile_df_head(profile, 7, sort_by_weight=False, percents=True, totals=True, n_decimals=3).drop(columns={"Voter Set",'Weight','Percent'},index='Total').transpose()

In [ ]:
profile_df_head(profile, 14, sort_by_weight=False, percents=True, totals=True, n_decimals=3).drop(columns={"Voter Set",'Weight','Percent'},index='Total').drop(index=range(0, 7), inplace=False).transpose()

In [ ]:
profile_df_head(profile, 21, sort_by_weight=False, percents=True, totals=True, n_decimals=3).drop(columns={"Voter Set",'Weight','Percent'},index='Total').drop(index=range(0, 14), inplace=False).transpose()

## Preference Tables

<font size="4">The individual ballots from an election are typically combined into one preference schedule or preference table. Each ranking from the election is shown in a column of the table. The row labelled weight gives the number of ballots with that ranking.</font>

## Example 1.2

<font size="4">Run the cell below. Here the ballots from example 1.1 have been combined into a preference table. Each column shows all the ballots that shared the same voter preference (Thus each index represents a group of ballots now). You should check example 1.1 to verify the weight of each column in the table matches the number of voters with that ballot. For instance, voters with ballot indices 9 and 15 from example 1.1 chose the ranking shown by ballot index 3, thus the weight is 2.
    </font>

In [ ]:
candidates=["A","B","C"]
profile = PreferenceProfile(ballots=ballots, candidates=candidates)
grouped_profile = profile.group_ballots()
#print(grouped_profile)
#print()
#print(grouped_profile.df.drop(columns="Voter Set").transpose().to_string())
profile_df_head(grouped_profile, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

## Ranked Ballots in the Real World

<font size="4">The ballots shown in example 1.1 are complete ballots. On a complete ballot one of the the candidates is listed on each ranking without any repeats. In actual elections many ballots are not complete. Voters make a lot of different choices.<font>

## Example 1.3

<font size="4">Shown below is a blank ranked ballot from Maine's 2020 general election for U.S. senate. Maine uses instant-runoff voting which requires ranked ballots. <font size="2">(sample ballot source: https://www.ellsworthmaine.gov/november3rd2020electionsampleballots/, Maine map source:https://ontheworldmap.com/usa/state/maine/)

<img src="SP images/Maine_senate_ballot_2020.png" width = 400 />&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;<img src="SP images/Maine.png" width = 200 />

<font size="4">Voter's are instructed to mark no more than one candidate for each choice (first through fifth) without selecting any candidate more than once. Voter's do not have to choose a candidate for each choice. Several examples of filled-in ballots are shown below.

<font size="4">Ballot 0:

<img src="SP images/Maine_ballot_complete.png" width = 400 />

<font size="4">Ballot 0 is called a complete ballot because the voter has chosen a candidate for each choice with no repeats.

<font size="4">Ballot 1:

<img src="SP images/Maine_ballot_undervote_1.png" width = 400 />

Ballot 1 is an undervote because the voter did not fill in all possible choices. Undervotes are counted.

<font size="4">Ballot 2:

<img src="SP images/Maine_ballot_undervote_2.png" width = 400 />

<font size="4">Ballot 2 is also an undervote.

Ballot 3:

<img src="SP images/Maine_ballot_overvote_1.png" width = 400 />

<font size="4">Ballot 3 is called an overvote, because the voter has chosen the same candidate more than once. In this case election officials would count the first choice vote for the candidate but not the other choices.

Ballot 4:

<img src="SP images/Maine_ballot_overvote_2.png" width = 400 />

<font size="4">Ballot 4 is also an overvote. In this case the voter's intentions are not clear. Election officials would count the voter's first choice but not any others.

Run the cell below to see how these ballots look when we use Votekit to combline them into a single table. We will use the following abbreviations:
- Susan Collins (C)
- Sara Gideon (G)
- Max Linn (L)
- Lisa Savage (S)
- Write-in (W)

In [ ]:
ballots1=[Ballot(ranking=[{"L"},{"C"},{"G"},{"S"},{"W"}]),Ballot(ranking=[{"G"}]),Ballot(ranking=[{'S'},{},{},{'G'}]),Ballot(ranking=[{"G"}]),
          Ballot(ranking=[{'G'},{'G'},{'G'},{'G'},{'G'}]),Ballot(ranking=[{"G"}]),Ballot(ranking=[{'S'},{'C','G'},{'L'}])]
candidates1=["C","G","L","S","W"]
profile = PreferenceProfile(ballots=ballots1, candidates=candidates1)

profile_df_head(profile, 7, sort_by_weight=False, percents=True, totals=True, n_decimals=3).drop(columns={"Voter Set",'Weight','Percent'},index='Total').transpose()

# Section 2: Plurality Elections 

<font size="4"> Steps to run a plurality election.

<font size="3">

    i. Each voter ranks the candidates.
    
    ii. Count the first-choice votes for each candidate.
    
    iii. The candidate that receives the most first-choice votes is declared the winner.


## Example 2.1

<font size="4"> Run the cell below to see the preference table from example 1.2.

In [ ]:
profile_df_head(grouped_profile, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4"> Now suppose this is the preference table for a plurality election. The table gives us each voters rankings of the candidates. So step (i) is completed. For step (ii) we need to determine the number of first-choice votes for each candidate.
- A receives 3+5=8 votes
- B receives 4+2=6 votes
- C receives 4+3=7 votes

<font size="4"> Since A received the most first-choice votes, A is declared the winner. A wins with 38% of the vote (8/21=0.38).

## Activity 2.2

<font size="4">Now you can practice tabulating the results of a plurality election.

<font size="4">a) Run the cell below to generate a preference table for an plurality election with candidates A, B, X, and Y and 501 ballots. Then compute the winner of the election.

In [ ]:
slate_to_candidates = {"Alpha": ["A", "B"], "Xenon": ["X", "Y"]}

# note that we include candidates with 0 support,
# and that our preference intervals will automatically rescale to sum to 1

preference_mapping = {
    "Alpha": {
        "Alpha": PreferenceInterval({"A": 0.7, "B": 0.3}),
        "Xenon": PreferenceInterval({"X": 0.01, "Y": 0.99}),
    },
    "Xenon": {
        "Alpha": PreferenceInterval({"A": 0.01, "B": 0.99}),
        "Xenon": PreferenceInterval({"X": 0.7, "Y": 0.3}),
    },
}


bloc_proportions = {"Alpha": 0.5, "Xenon": 0.5}

# assume that each bloc is 90% cohesive
# we'll discuss exactly what that means later
cohesion_mapping = {
    "Alpha": {"Alpha": 0.9, "Xenon": 0.1},
    "Xenon": {"Xenon": 0.9, "Alpha": 0.1},
}

config = bg.BlocSlateConfig(
    n_voters=501,
    preference_mapping=preference_mapping,
    bloc_proportions=bloc_proportions,
    slate_to_candidates=slate_to_candidates,
    cohesion_mapping=cohesion_mapping,
)

# by using the by_bloc generator we can see which ballots came from which blocs of voters
profile_dict = bg.name_pl_profiles_by_bloc_generator(config)
# print("The ballots from Alpha voters\n", profile_df_head(profile_dict["Alpha"], 10))

# print("The ballots from Xenon voters\n", profile_df_head(profile_dict["Xenon"], 10))

# to create the aggregate profile, we just sum the profiles in the dictionary
agg_profile = profile_dict["Alpha"] + profile_dict["Xenon"]
# print("Aggregated ballots\n", profile_df_head(agg_profile, 10))

display(profile_df_head(agg_profile, 24, sort_by_weight=False, percents=False, totals=True, n_decimals=3).drop(columns="Voter Set").drop(index='Total').transpose())

<font size="4">b) Run the cell below to check your answer to part (a).

In [ ]:
plurality_results(agg_profile)

<font size="4">Part (a) generates a different preference table each time you run it. Repeat activity for more practice.

# Section 3: Fairness criterion: Independence of Irrelevant Alternatives (IIA)

<font size="4">_If a losing candidate in an election is removed from all of the ballots and the election is recounted the winning candidate should not change._

<font size="4">This fairness criterian helps social choice researchers determine the effect that minor candidates have when using a particular voting system. If the election outcome is different after removing the minor candidate, that candidate is often referred to as a spoiler. The severity of the spoiler effect differs depending on the voting system. Election methods that minimize the impact of spoiler candidates are considered preferable by many social choice researchers.

## Example 3.1

<font size="4">Let's see if the plurality election from example 2.1 satisfies the Independence of Irrelevant Alternatives criterion. 

<font size="4">a) Run the cell below to compute the results of the election. (It should be the same as what we got in Example 2.1.

In [ ]:
plurality_results(grouped_profile)

<font size="4">b) If we can find a single instance of removing a losing candidate that changes the winner of the election, the election does not satisfy the IIA criteria.

<font size="4">We will begin by removing a losing candidate from the ballots. This requires that we shift up lower ranked candidates to fill any empty spots in rankings 1 and 2. Then we will re-run the plurality election. 

<font size="4">Let's start by removing losing candidate C from the ballots. Compare to the preference table in example 2.1 to see how the ballots have changed. Run the cell below.

In [ ]:
profile_C=remove_and_condense_rank_profile("C", grouped_profile)
profile_df_head(profile_C, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">Compute the results of the election using the modified preference table by running the cell below.

In [ ]:
plurality_results(profile_C)

print()
if Plurality(profile=profile_C, m=1).get_elected()==Plurality(profile=grouped_profile, m=1).get_elected():
    print("A still wins the election.")
else:
    print("Since A was not elected, the election from example 2.1 did not satisfy the Independence of Irrelevant Alternatives criterion.")

<font size="4">Now remove losing candidate B from the ballots. Compare to the preference table in example 2.1 to see how the ballots have changed.

In [ ]:
profile_B=remove_and_condense_rank_profile("B", grouped_profile)
profile_df_head(profile_B, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">Compute the results of the election using the modified preference table by running the cell below.

In [ ]:
plurality_results(profile_B)

print()
if Plurality(profile=profile_B, m=1).get_elected()==Plurality(profile=grouped_profile, m=1).get_elected():
    print("A still wins the election.")
else:
    print("Since A was not elected, the election from example 2.1 did not satisfy the Independence of Irrelevant Alternatives criterion.")

## Activity 3.2

<font size="4">Now it's your turn. Suppose a plurality election is run with candidates D, N, P and S. In this activity you will determine whether this election satisfies the Independence of Irrelevant Alternatives criterion.

<font size="4"> a) Run the cell below to see the preference table for the election. (See if you can determine the winner before completing part (b).)

In [ ]:
ballots_activity_2={Ballot(ranking=[{"D"},{"N"},{"P"},{"S"}],weight=8),Ballot(ranking=[{"P"},{"S"},{"N"},{"D"}],weight=5),
             Ballot(ranking=[{"N"},{"D"},{"S"},{"P"}],weight=10),Ballot(ranking=[{"P"},{"N"},{"S"},{"D"}],weight=9),
             Ballot(ranking=[{"N"},{"P"},{"S"},{"D"}],weight=6),Ballot(ranking=[{"S"},{"D"},{"P"},{"N"}],weight=12),
             Ballot(ranking=[{"P"},{"N"},{"S"},{"D"}],weight=7)}
candidates_activity_2={"D","N","P","S"}
profile_activity_2 = PreferenceProfile(ballots=ballots_activity_2, candidates=candidates_activity_2)
profile_df_head(profile_activity_2, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">b) Run the follwoing cell to see the results of the plurality election.

In [ ]:
plurality_results(profile_activity_2)

<font size="4">c) Run the following cell to remove one of the losing candidates from the ballots. The output will show the modified preference table.

In [ ]:
cand=input('Enter a losing candidate.')
if cand=='P':
    print('Invalid entry. Enter a losing candidate. Try again')
else:
    profile_activity_2_rem=remove_and_condense_rank_profile(cand, profile_activity_2)
    display(profile_df_head(profile_activity_2_rem, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose())

<font size="4">d) Now you need to determine if the candidate that was removed in part (d) changes the outcome of the election. If it does, the election does not satisfy the IIA criterion. Run the following cell. The output will show the results of the plurality election after removing the candidate from part (d).

In [ ]:
plurality_results(profile_activity_2_rem)

print()
if Plurality(profile=profile_activity_2_rem, m=1).get_elected()==Plurality(profile=profile_activity_2, m=1).get_elected():
    print("P still wins the election.")
else:
    print("Since P was not elected, the election (from part (a)) did not satisfy the Independence of Irrelevant Alternatives criterion.")
print()

<font size="4">f) If removing the candidate in (d) did not change the outcome of the election, repeat parts (d) and (e) until you find a losing candidate that does. If no such candidate can be found, the election does satisfy the IIA criterion.

## Plurality and two-party systems

<font size="4">Duverger's law says that systems that use plurality elections with a single winner will tend toward a two party system. [Schlesinger, Schlesinger 2006]

As we can see from example 3.1 and activity 3.2 losing candidates can determine the winner of a plurality election making them potential spoilers. This incentivizes voters to abandon a preferred candidate and instead vote for a lower preference candidate polling in the top two in order to avoid "wasting their vote" (this is known as strategic or insincere voting). This drives the plurality system towards a two-party system.

# Section 4: Instant-runoff Elections 
<font size="4">Proponents of multi-party democracy have advocated for alternatives to plurality elections. Instant-runoff has been proposed as one such alternative. Versions of instant-runoff have been adopted in the states of Alaska and Maine. It is also used in numerous municipalities including San Francisco and Oakland. Australia uses it for elections to its house of representatives.

An instant-runoff election is an election where the candidate with the least first-choice votes is eliminated and that candidate's votes are redistributed to the remaining candidates based upon their voters' rankings. This process is repeated until a candidate has a majority of the votes and is declared the winner.

<font size="4">Instant run-off voting (IRV) is also referred is often referred to as ranked choice voting. However, there are many elections that use a ranked ballot but a method other than instant-runoff voting.

<font size="4">Before giving the steps to conduct an instant-runoff election let's just look at an example to see how the process works.

## Example 4.1

<font size="4">In this example we will tabulate the results of an instant-runoff election by completing each part.

<font size="4">a) Run the following cell to see the preference table for the instant-runoff election involving candidates A, B, C, D and E.

In [ ]:
ballots_test={Ballot(ranking=[{"A"},{"D"},{"E"},{"C"},{"B"}],weight=12),Ballot(ranking=[{"C"},{"B"},{"D"},{"A"},{"E"}],weight=9),
             Ballot(ranking=[{"B"},{"E"},{"D"},{"A"},{"C"}],weight=8),Ballot(ranking=[{"D"},{"C"},{"B"},{"A"},{"E"}],weight=7),
             Ballot(ranking=[{"E"},{"D"},{"B"},{"C"},{"A"}],weight=10),Ballot(ranking=[{"D"},{"B"},{"C"},{"A"},{"E"}],weight=6),
             Ballot(ranking=[{"C"},{"A"},{"B"},{"E"},{"D"}],weight=5)}
candidates_test={"A","B","C","D","E"}
profile = PreferenceProfile(ballots=ballots_test, candidates=candidates_test)
grouped_profile_IRV = profile.group_ballots()
profile_df_head(grouped_profile_IRV, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4"> b) Tabulate the number of first-choice votes each candidate received and determine whether any candidate received a majority. Then run the cell below to check your results.

In [ ]:
election_activity_IRV = IRV(profile=grouped_profile_IRV)
#election_activity_IRV = Plurality(profile=grouped_profile_IRV)
print()
y=election_activity_IRV.election_states[0].scores
print("Vote Totals")
for candidate in y:
    print(candidate, int(y[candidate]))
print()
total=0
for candidate in y:
    total=total+y[candidate]
for candidate in y:
    print("Candidate ", candidate, "received ", round(y[candidate]/total*100,2), "% of the vote.")
for candidate in y:
    if y[candidate]/total*100 > 50:
        print()
        print("Candidate ", candidate, "received a majority of the votes and won the election.")
        break
else:
    print()
    print("No candidate has a majority. We must continue tabulating results to find winner.")

<font size="4">c) Make a copy of the preference table from (a). Cross off the candidate from (b) with the fewest votes. Make an updated preference table by "shifting up" the lower candidates. Run the cell below to check your answer.  

In [ ]:
w = election_activity_IRV.election_states[1].eliminated
profile_2=remove_and_condense_rank_profile(w[0], grouped_profile_IRV)
profile_df_head(profile_2, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4"> d) From the preference table in (c) tabulate the number of first-choice votes each candidate has and determine whether any candidate now has a majority. Then run the cell below to check your results.

In [ ]:
election_activity_IRV_2 = IRV(profile=profile_2)
print()
y=election_activity_IRV_2.election_states[0].scores
print("Vote Totals")
for candidate in y:
    print(candidate, int(y[candidate]))
print()
total=0
for candidate in y:
    total=total+y[candidate]
for candidate in y:
    print("Candidate ", candidate, "received ", round(y[candidate]/total*100,2), "% of the vote.")
for candidate in y:
    if y[candidate]/total*100 > 50:
        print()
        print("Candidate ", candidate, "received a majority of the votes and won the election.")
        break
else:
    print()
    print("No candidate has a majority. We must continue tabulating results to find winner.")

<font size="4">e) Make a copy of the preference table from (c). Cross off the candidate from (d) with the fewest votes. Make an updated preference table by "shifting up" the lower candidates. Run the cell below to check your answer.  

In [ ]:
w = election_activity_IRV_2.election_states[1].eliminated
profile_3=remove_and_condense_rank_profile(w[0], profile_2)
profile_df_head(profile_3, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4"> f) From the preference table in (e) tabulate the number of first-choice votes each candidate has and determine whether any candidate now has a majority. Then run the cell below to check your results.

In [ ]:
election_activity_IRV_3 = IRV(profile=profile_3)
print()
y=election_activity_IRV_3.election_states[0].scores
print("Vote Totals")
for candidate in y:
    print(candidate, int(y[candidate]))
print()
total=0
for candidate in y:
    total=total+y[candidate]
for candidate in y:
    print("Candidate ", candidate, "received ", round(y[candidate]/total*100,2), "% of the vote.")
for candidate in y:
    if y[candidate]/total*100 > 50:
        print()
        print("Candidate ", candidate, "received a majority of the votes and won the election.")
        break
else:
    print()
    print("No candidate has a majority. We must continue tabulating results to find winner.")

<font size="4">g) Make a copy of the preference table from (e). Cross off the candidate from (f) with the fewest votes. Make an updated preference table by "shifting up" the lower candidates. Run the cell below to check your answer.  

In [ ]:
w = election_activity_IRV_3.election_states[1].eliminated
profile_4=remove_and_condense_rank_profile(w[0], profile_3)
profile_df_head(profile_4, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

In [ ]:
election_activity_IRV_4 = IRV(profile=profile_4)
print()
y=election_activity_IRV_4.election_states[0].scores
print("Vote Totals")
for candidate in y:
    print(candidate, int(y[candidate]))
print()
total=0
for candidate in y:
    total=total+y[candidate]
for candidate in y:
    print("Candidate ", candidate, "received ", round(y[candidate]/total*100,2), "% of the vote.")
for candidate in y:
    if y[candidate]/total*100 > 50:
        print()
        print("Candidate ", candidate, "received a majority of the votes and won the election.")
        break
else:
    print()
    print("No candidate has a majority. We must continue tabulating results to find winner.")

<font size="6"> Steps to run an instant-runoff election.

<font size="3">

    i. Each voter ranks the candidates.
    
    ii. Round 1: Count the first-choice votes for each candidate. If a candidate receives a majority of first-choice votes that candidate is declared the winner and the election is over. Otherwise continue to next step.
    
    iii. Round 2: Eliminate the candidate from the previous round that received the fewest first-choice votes by crossing them off each ballot. Redistribute the eliminated candidates votes by assigning them to the next highest choice (this is now considered the first choice) on each of those ballots. Count the first-choice votes for each candidate. If a candidate now receives a majority of first-choice votes that candidate is declared the winner and the election is over. Otherwise continue to next step.
    
    iv. Round 3 and onward: Repeat step (iii) until the election is complete.


<font size="4">Note that these are steps one uses when computing the results by "hand". Another equivalent method is to remove the eliminated candidate from each ballot and shift up any candidates that were ranked below the eliminated candidate. Then count all the first-choice votes.

## Activity 4.2

<font size="4">In this activity you will tabulate the results of an instant-runoff election and check your answer using Votekit. Then you will test to see if IRV may violate the Independence of Irrelevant Alternatives criterion.

<font size="4">a) Run the cell below to generate a preference table for the instant-runoff election.

In [ ]:
slate_to_candidates = {"Alpha": ["A", "B"], "Xenon": ["X", "Y"]}

# note that we include candidates with 0 support,
# and that our preference intervals will automatically rescale to sum to 1

preference_mapping = {
    "Alpha": {
        "Alpha": PreferenceInterval({"A": 0.6, "B": 0.4}),
        "Xenon": PreferenceInterval({"X": 0.3, "Y": 0.7}),
    },
    "Xenon": {
        "Alpha": PreferenceInterval({"A": 0.3, "B": 0.7}),
        "Xenon": PreferenceInterval({"X": 0.6, "Y": 0.4}),
    },
}


bloc_proportions = {"Alpha": 0.5, "Xenon": 0.5}

# assume that each bloc is 90% cohesive
# we'll discuss exactly what that means later
cohesion_mapping = {
    "Alpha": {"Alpha": 0.9, "Xenon": 0.1},
    "Xenon": {"Xenon": 0.9, "Alpha": 0.1},
}

config = bg.BlocSlateConfig(
    n_voters=207,
    preference_mapping=preference_mapping,
    bloc_proportions=bloc_proportions,
    slate_to_candidates=slate_to_candidates,
    cohesion_mapping=cohesion_mapping,
)

# by using the by_bloc generator we can see which ballots came from which blocs of voters
profile_dict = bg.name_pl_profiles_by_bloc_generator(config)
# print("The ballots from Alpha voters\n", profile_df_head(profile_dict["Alpha"], 10))

# print("The ballots from Xenon voters\n", profile_df_head(profile_dict["Xenon"], 10))

# to create the aggregate profile, we just sum the profiles in the dictionary
agg_profile_IRV_temp_1 = profile_dict["Alpha"] + profile_dict["Xenon"]
agg_profile_IRV = agg_profile_IRV_temp_1.group_ballots()

# print(agg_profile_IRV)

# display(profile_df_head(agg_profile_IRV, 24, sort_by_weight=False, percents=False, totals=True, n_decimals=3).drop(columns="Voter Set").drop(index='Total').transpose())
for i in range(0, len(agg_profile_IRV.ballots), 12):
    display(profile_df_head(agg_profile_IRV, i + 12, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").drop(index=range(0, i), inplace=False).transpose())

<font size="4">b) Compute the results of the election using the instant-runoff method. This will be similar to Example 4.1. 

<font size="3">Helpful hints:

    i. Follow the "Steps to run an instant-runoff election" following Example 4.1.
    ii. You should copy the preference table so that you can cross-off candidates as they are eliminated. 
    iii. Find the total number of votes. This way you can check to see if a candidate has a majority in any round. 
    iv. Write out the vote totals for each round. 
    v. To compute each round you can use the totals from the previous round, just add on the redistributed votes from the eliminated candidate. 

<font size="4">c) Check your answer to (c) by running the cell below.

In [ ]:
election_activity_IRV_B = IRV(profile=agg_profile_IRV)
display(election_activity_IRV_B)

<font size="4">d) Get more details on each round of the election run the cell below.

In [ ]:
IRV_results_complete(agg_profile_IRV)

<font size="4">e) Let's try to find out if the election violates the Independence of Irrelevant Alternatives criterion. Run the following cell to remove one of the losing candidates from the ballots. The output will show the modified preference table.

In [ ]:
cand=input("Enter a losing candidate.")
m=IRV(agg_profile_IRV).get_elected()[0]
n,=m

if cand == n:
    print('Invalid entry. Enter a losing candidate. Try again')
else:
    agg_profile_IRV_2=remove_and_condense_rank_profile(cand, agg_profile_IRV)
    display(profile_df_head(agg_profile_IRV_2, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose())

<font size="4">f) Now determine if the candidate that was removed in part (e) changes the outcome of the election. If it does, the election does not satisfy the IIA criterion. Run the following cell. The output will show the results of the plurality election after removing the candidate from part (e).

In [ ]:
IRV_results_complete(agg_profile_IRV_2)

print()
if IRV(agg_profile_IRV_2).get_elected()==IRV(agg_profile_IRV).get_elected():
    print(n, "still wins the election.")
else:
    print("Since", n, " was not elected, the election from(c) did not satisfy the Independence of Irrelevant Alternatives criterion.")
print()

<font size="4">g) Repeat parts (e) and (f) until the outcome of the election changes or you have tried removing each of the losing candidates.

<font size="4">h) What do you think? Is it possible for an instant-runoff election to violate the Independence of Irrelevenat Alternatives criteria? Run the cell below to answer.

In [ ]:
while True:
        answer=input("Is it possible for an instant-runoff election to violate the Independence of Irrelevant Alternatives criterion? (y/n)")
        if answer == 'y' or answer == 'n':
            break
        else:
            print("Invalid input")
if answer == "y":
    print("Correct. IRV may violate the Independence of Irrelevant Alternatives criterion.")
else:
    print("Incorrect. It is possible for an instant-runoff election to violate the Independence of Irrelevant Alternatives criterion.")
    print()
    print("Go back to part (a) and generate a new preference table and repeat the experiment until you find an exaample where the election violates IIA. It is possible.")



<font size="4"> Now let's take a look a real-world instant run-off election.

## Example 4.3

<font size="4"> In 2009 the mayoral election in Burlington, Vermont used the instant-runoff method. There were 5 candidates: Bob Kiss (BK), Andy Montroll (AM), James Simpson (JS), Dan Smith (DS), and Kurt Wright (KW). Write in candidates were also allowed. 

<font size="4">Ballots in real-world elections look different from the hypothetical elections we have looked at so far. For the purpose of this example six overvotes were removed from ballots. And there were 128 instances of write in candidates that were also removed from the ballots. The preference table is still a good approximation of the set of ballots that were actually cast in the election. Removing these ballots does not cause any significant changes in the results. Voters could rank up to 5 candidates. You will notice that there are many undervotes.

<font size="4">a) Run the cell below to view the preference table for this election.


In [ ]:
candidates = ["BK", "AM", "JS", "DS", "KW", "WI"]
burlington_ballots = []

with open("CVR files/burlington_mayor.csv", newline="", encoding="utf-8-sig") as ballotfile:
    for row in csv.reader(ballotfile):
        if not row or row[0].strip().lower() == "count":
            continue

        ranking = [
            {candidate.strip()}
            for candidate in row[1:]
            if candidate.strip()
        ]

        ballot =Ballot(
            ranking=ranking,
            weight=float(row[0])
        )
        burlington_ballots.append(ballot)

profile_burlington = PreferenceProfile(
    ballots=burlington_ballots,
    candidates=candidates
)

profile_burlington_clean_temp_1=remove_cand_rank_profile("WI", profile_burlington, remove_empty_ballots = False)
profile_burlington_clean_temp_2 = condense_rank_profile(profile_burlington_clean_temp_1)

profile_burlington_clean = profile_burlington_clean_temp_2.group_ballots()

print(profile_burlington_clean)
for i in range(0, 260, 15):
    display(profile_df_head(profile_burlington_clean, i + 15, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").drop(index=range(0, i), inplace=False).transpose())

<font size="4">b) To see the results of the election run the cell below.

In [ ]:
election_burlington = IRV(profile=profile_burlington_clean)
display(election_burlington)
print()
print("Note that Votekit computed a 5th round which had we computed the results by hand, we would have stopped after the 4th round since there were only 4 candidates remaining. The 5th round is a result of the way Votekit computes the IRV election results.")

<font size="4">c) Get more details on each round of the election run the cell below.

<font size="4"> You will see that as candidates are eliminated the total number of votes in the following round decreases. When a candidate is eliminated some of the undervote ballots will not have a preference to whom the vote can be redistributed. These are referred to as exhausted ballots.

In [ ]:
IRV_results_undervotes(profile_burlington_clean)

# Section 5: Fairness criterion: Monotonicity

<font size="4">_Suppose we have two ballot preference tables A and B. The preference tables are identical except that candidate X is ranked higher on some ballots in preference table B. If X wins the election with preference table A, then X should also win the election with preference table B_.

<font size="4"> In other words an election is held, candidate X wins. If some of the voters had ranked candidate X even higher, candidate X should still win the election. Getting more support should never hurt a candidate. Many social choice researchers consider voting systems that violate monotonicity to be seriously flawed [Falsenthal, Tideman 2014].

## Example 5.1

<font size="4"> Let's take a look at an election that shows an election that fails to satisfy the monotonicity criteria. Suppose an instant-runoff election is held with candidates A, B and C. 

<font size="4"> a) Run the cell below to view the preference table for this election. 

In [ ]:
ballots_mono=[Ballot(ranking=[{"A"},{"B"},{"C"}], weight=13),Ballot(ranking=[{"B"},{"A"},{"C"}], weight=9),Ballot(ranking=[{"C"},{"B"},{"A"}], weight=8)]
candidates=["A","B","C"]
profile_mono = PreferenceProfile(ballots=ballots_mono, candidates=candidates)

profile_df_head(profile_mono, 7, sort_by_weight=False, percents=True, totals=True, n_decimals=3).drop(columns={"Voter Set",'Percent'},index='Total').transpose()

<font size="4"> b) Using the preference table in (a) compute the outcome of the instant run-off election. Check your answer to by running the cell below.

In [ ]:
IRV_results_complete(profile_mono)

<font size="4"> c) Now suppose that two ballots were recorded with ranking ABC should have been recorded as CAB. Run the cell below to view the corrected preference table for this election.

In [ ]:
ballots_mono_corrected=[Ballot(ranking=[{"A"},{"B"},{"C"}], weight=11),Ballot(ranking=[{"B"},{"A"},{"C"}], weight=9),Ballot(ranking=[{"C"},{"B"},{"A"}], weight=8),Ballot(ranking=[{"C"},{"A"},{"B"}], weight=2)]
candidates=["A","B","C"]
profile_mono_corrected = PreferenceProfile(ballots=ballots_mono_corrected, candidates=candidates)

profile_df_head(profile_mono_corrected, 7, sort_by_weight=False, percents=True, totals=True, n_decimals=3).drop(columns={"Voter Set",'Percent'},index='Total').transpose()

<font size="4"> d) Using the preference table in (c) compute the outcome of the instant run-off election. Check your answer to by running the cell below.

In [ ]:
IRV_results_complete(profile_mono_corrected)

<font size="4"> Unfortunately Example 5.1 shows that instant-runoff voting may violate monotonicty. 

## Activity 5.2

<font size="4"> In this activity you will show that the 2009 the mayoral instant-runoff election in Burlington, Vermont did not satisfy monotonicity. 

<font size="4"> a) We only need to examine the election tabulation after James Simpson (JS) and Dan Smith (DS) were eliminated, leaving only Bob Kiss (BK), Andy Montroll (AM), and Kurt Wright (KW). Run the cell below to view the preference table after the candidates are removed.

In [ ]:
candidates = ["BK", "AM", "JS", "DS", "KW", "WI"]
burlington_ballots = []

with open("CVR files/burlington_mayor.csv", newline="", encoding="utf-8-sig") as ballotfile:
    for row in csv.reader(ballotfile):
        if not row or row[0].strip().lower() == "count":
            continue

        ranking = [
            {candidate.strip()}
            for candidate in row[1:]
            if candidate.strip()
        ]

        ballot =Ballot(
            ranking=ranking,
            weight=float(row[0])
        )
        burlington_ballots.append(ballot)

profile_burlington = PreferenceProfile(
    ballots=burlington_ballots,
    candidates=candidates
)

profile_burlington_clean_3=remove_cand_rank_profile({"DS","JS","WI"}, profile_burlington, remove_empty_ballots = False)
profile_burlington_clean_4 = condense_rank_profile(profile_burlington_clean_3)

grouped_burlington = profile_burlington_clean_4.group_ballots()
#print(grouped_burlington.df)

display(profile_df_head(grouped_burlington, 100, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose())


<font size="4"> b) Run the cell below to see the results of the election. (These are the same as what you saw in Example 4.3.)

In [ ]:
IRV_results_undervotes(grouped_burlington)

<font size="4"> c) In order to show that the election failed monotonicity we need to show that the winner of the election, Bob Kiss, will lose the election when some of the ballots are changed to give him higher rankings and the other ballots remain the same.

<font size="4"> If you switch some of the bullet votes for Kurt Wright to have a ranking the ranking Bob Kiss > Kurt Wright and some of the ballots with ranking Kurt Wright > Bob Kiss > Andy Montroll to Bob Kiss > Kurt Wright > Andy Montroll you can change the winner of the election.

<font size="4"> You may want to look at the preference table in (a) first to help you decide how many votes need to be switched.

<font size="4"> Run the cell below to alter the ballots and get the new preference table.

In [ ]:
while True:
        x = float(input("Enter the number of voters who will switch ranking from KW to BK>KW (a value between 0 and 1289): "))
        if 0 <= x <= 1289:
            break
        else:
            print("Invalid input")
while True:
        y = float(input("Enter the number of voters who will switch ranking from KW>BK>AM to BK>KW>AM (a value between 0 and 342): "))
        if 0 <= y <= 342:
            break
        else:
            print("Invalid input")            

#new_ballot = Ballot(ranking=[{"BK"}, {"KW"}, {"AM"}], weight=x)

updated_ballots = list(grouped_burlington.ballots)  # if this attribute exists in your version
#updated_ballots.append(new_ballot)
updated_ballots[0] = Ballot(ranking=[{"KW"}], weight=1289 - x)
updated_ballots[4]=Ballot(ranking=[{"KW"}, {"BK"}, {"AM"}], weight=342-y)
updated_ballots[8]=Ballot(ranking=[{"BK"}, {"KW"}], weight=118+x)
updated_ballots[9]=Ballot(ranking=[{"BK"}, {"KW"}, {"AM"}], weight=252+y)

grouped_burlington_2 = PreferenceProfile(
    ballots=updated_ballots,
    candidates=grouped_burlington.candidates
)

display(profile_df_head(grouped_burlington_2, 100, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose())


<font size="4"> d) Run the cell below to see the results of the election with the preference table from (c).. 

In [ ]:
IRV_results_undervotes(grouped_burlington_2)

print()

if IRV(grouped_burlington_2).get_elected()==IRV(grouped_burlington).get_elected():
    print("BK still won the election. Give parts (c) and (d) another try.")
else:
    print("BK did not win the election. You have successfully demonstrated that the IRV election method does not satisfy the monotonicity criterion.")

# Section 6: Pairwise Comparison Elections

<font size="4"> These methods determine the winner of the election by conducting a round-robin tournament between all of the candidates. These are also referred to also Condorcet methods (named after the Nicolas de Condorcet). We will focus on one particular Condorcet method known as Copeland's method. However, we first need to know how pairwise comparisons of candidates are determined.

## Comparing candidates pairwise

<font size="4">Suppose we have a preference table with rankings for several candidates: A, B, C, etc. For a given pair of candidates, say A and B. Check each ballot. If A has the higher ranking it receives a vote, otherwise B receives the vote. Total up the votes for each candidate. The candidate with the most votes wins that pairing. Ties are possible.

## Steps to run a Copeland pairwise comparison election

<font size="3">

    i. Each voter ranks the candidates.
    
    ii. Determine all possible pairs of candidates.
    
    iii. Choose a pair of candidates (not already selected). The candidate that wins the pairing receives 1 point and the other candidate receives 0 points. If the pairing was a tie each candidate receives 1/2 point.
    
    iv. Repeat step iii until all pairings have been tabulated.

    v. Tally the total points for each candidate. The candidate with the most points is the winner.


## Example 6.1

<font size="4"> Let's compute the results of a Copeland pairwise comparison election involving candidates A, B, C and D.

<font size="4">a) Run the cell below to view the preference table for the election.

In [ ]:
ballots_copeland_1={Ballot(ranking=[{"A"},{"B"},{"C"},{"D"}],weight=13),Ballot(ranking=[{"C"},{"D"},{"B"},{"A"}],weight=7),
             Ballot(ranking=[{"B"},{"A"},{"D"},{"C"}],weight=15),Ballot(ranking=[{"C"},{"B"},{"D"},{"A"}],weight=8),
             Ballot(ranking=[{"B"},{"C"},{"D"},{"A"}],weight=11),Ballot(ranking=[{"D"},{"A"},{"C"},{"B"}],weight=10),
             Ballot(ranking=[{"C"},{"B"},{"D"},{"A"}],weight=12)}
candidates_copeland_1={"A","B","C","D"}
profile_copeland_1 = PreferenceProfile(ballots=ballots_copeland_1, candidates=candidates_copeland_1)
profile_df_head(profile_copeland_1, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">b) Step (i) is complete. For step (ii) we need to determine all possible pairs of candidates. Go ahead and try to list the pairs. Then run the cell below to check your answer.

In [ ]:
pwc_graph_copeland_1 = PairwiseComparisonGraph(profile_copeland_1)
pwc_graph_copeland_1.pairwise_dict.keys()

for i in range (0, len(pwc_graph_copeland_1.pairwise_dict.keys())):
    print(list(pwc_graph_copeland_1.pairwise_dict.keys())[i])

<font size="4">c) For step (iii) we will start with the (A, B) pairing, detemine who wins and award the point.

<font size="4">On ballots with indexes 0 and 2 candidate A is ranked higher than candidate B. Those votes go to candidate A.
- A receives 13+10=23 votes.

<font size="4">On ballots with indexes 1, 3, 4, 5 and 6 candidate B is ranked higher than candidate A. Those votes go to candidate B.
- B receives 15+11+8+7+12=53 votes.

<font size="4"> Candidate B wins and gets 1 point and A receives 0 points.

<font size="4">d) For steps (iv) we will repeat step (iii) for the (A, C) pairing, detemine who wins and award the point.

<font size="4">On ballots with indexes 0, 1 and 2 candidate A is ranked higher than candidate C. Those votes go to candidate A.
- A receives 13+15+10=38 votes.

<font size="4">On ballots with indexes 3, 4, 5 and 6 candidate C is ranked higher than candidate A. Those votes go to candidate C.
- C receives 11+8+7+12=38 votes.

<font size="4"> Candidate A and C are tied. Each candidate gets 0.5 points.

<font size="4">e) For the remaining pairs, determine who wins and award the point. Run the cell below and check your work.

In [ ]:
for (cand1, cand2), (cand1_votes, cand2_votes) in pwc_graph_copeland_1.pairwise_dict.items():
    if cand1_votes > cand2_votes:
        print(f"{cand1} receives {cand1_votes} votes and {cand2} receives {cand2_votes} votes.  {cand1} receives 1 point.")
    elif cand2_votes > cand1_votes:
        print(f"{cand2} receives {cand2_votes} votes and {cand1} receives {cand1_votes} votes.  {cand2} receives 1 point.")
    else:
        print(f"{cand2} receives {cand2_votes} votes and {cand1} receives {cand1_votes} votes.  Each candidate receives 0.5 points.")

<font size="4">f) Now complete step (v). Tally the votes and determine the total votes for each candidate and the winner of the election. Run the cell below to check your answer.

In [ ]:
copeland_scores_1 = copeland_score(profile_copeland_1)
print("Copeland scores:", copeland_scores_1)

copeland_results(profile_copeland_1)

<font size="4">g) Run the following cell to see the pairwise comparison graph illustrating the results of the election. Each node represents a candidate. The arrows point away from the pair winner and toward the pair loser. The number along the arrow indicates the margin of victory. Notice that arrow connecting the nodes between A and C points in both directions. This indicates the tie between A and C. 

In [ ]:
pwc_graph_copeland_1 = PairwiseComparisonGraph(profile_copeland_1)
pwc_graph_copeland_1.draw()

## Condorcet candidate

<font size="4"> _A Condorcet candidate is a candidate who wins all one-on-one (pairwise) comparisons versus each other candidate._

<br/>
<font size="4">Candidate B in Example 6.1 was a Condorcet candidate. Candidate B won when paired directly against candidates A, C and D.

## Example 6.2

<font size="4">Let's look at some Copeland pairwise comparison elections and determine if there was a Condorcet candidate.

<font size="4">a) Run the cell below to see the preference table for candidates A, B, C, D and E. (This is the same preference table that was used in the instant-runoff election in example 4.1.)

In [ ]:
#copeland example

ballots_test_copeland_2a={Ballot(ranking=[{"A"},{"D"},{"E"},{"C"},{"B"}],weight=12),Ballot(ranking=[{"C"},{"B"},{"D"},{"A"},{"E"}],weight=9),
             Ballot(ranking=[{"B"},{"E"},{"D"},{"A"},{"C"}],weight=8),Ballot(ranking=[{"D"},{"C"},{"B"},{"A"},{"E"}],weight=7),
             Ballot(ranking=[{"E"},{"D"},{"B"},{"C"},{"A"}],weight=10),Ballot(ranking=[{"D"},{"B"},{"C"},{"A"},{"E"}],weight=6),
             Ballot(ranking=[{"C"},{"A"},{"B"},{"E"},{"D"}],weight=5)}
candidates_test={"A","B","C","D","E"}
profile_copeland_2a = PreferenceProfile(ballots=ballots_test_copeland_2a, candidates=candidates_test)
grouped_profile_copeland_2a = profile_copeland_2a.group_ballots()
profile_df_head(grouped_profile_copeland_2a, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">Run the cell below to see the results of the election and any Condorcet candidate.

In [ ]:
copeland_scores_2a = copeland_score(grouped_profile_copeland_2a)
print()
print("Copeland scores:", copeland_scores_2a)
print()
copeland_results(profile_copeland_2a)
print()
pwc_graph_copeland_2a = PairwiseComparisonGraph(profile_copeland_2a)

# condorcet winner
if pwc_graph_copeland_2a.has_condorcet_winner() == True:
    print("The Condorcet candidate is:", pwc_graph_copeland_2a.get_condorcet_winner())
else:
    print("There is no Condorcet candidate.")

<font size="4">Run the cell below to see the pairwise comparison graph illustrating the results of the election.

In [ ]:
pwc_graph_copeland_2a = PairwiseComparisonGraph(grouped_profile_copeland_2a)
pwc_graph_copeland_2a.draw()

<font size="4">b) Run the cell below to see the preference table for candidates W, X, Y and Z. 

In [ ]:
ballots_activity_copeland_2b={Ballot(ranking=[{"W"},{"X"},{"Y"},{"Z"}],weight=10),Ballot(ranking=[{"Y"},{"Z"},{"X"},{"W"}],weight=10),
             Ballot(ranking=[{"X"},{"Z"},{"W"},{"Y"}],weight=10),Ballot(ranking=[{"Y"},{"X"},{"Z"},{"W"}],weight=10),
             Ballot(ranking=[{"X"},{"Y"},{"Z"},{"W"}],weight=10),Ballot(ranking=[{"Z"},{"W"},{"Y"},{"X"}],weight=10),
             Ballot(ranking=[{"Y"},{"X"},{"Z"},{"W"}],weight=10),Ballot(ranking=[{"Z"},{"X"},{"Y"},{"W"}],weight=20)}
candidates_activity_copeland_2b={"W","X","Y","Z"}
profile_activity_copeland_2b = PreferenceProfile(ballots=ballots_activity_copeland_2b, candidates=candidates_activity_copeland_2b)
profile_df_head(profile_activity_copeland_2b, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">Run the cell below to see the results of the election and any Condorcet candidate.

In [ ]:
copeland_scores_2b_cond = copeland_score(profile_activity_copeland_2b)
print()
print("Copeland scores:", copeland_scores_2b_cond)
print()
copeland_results(profile_activity_copeland_2b)
print()
pwc_graph_copeland_2b = PairwiseComparisonGraph(profile_activity_copeland_2b)

# condorcet winner
if pwc_graph_copeland_2b.has_condorcet_winner() == True:
    print("The Condorcet candidate is:", pwc_graph_copeland_2b.get_condorcet_winner())
else:
    print("There is no Condorcet candidate.")

<font size="4">Run the cell below to see the pairwise comparison graph illustrating the results of the election.

In [ ]:
pwc_graph_copeland_2b = PairwiseComparisonGraph(profile_activity_copeland_2b)
pwc_graph_copeland_2b.draw()

<font size="4">c) Run the cell below to see the preference table for candidates J, K, L and M.

In [ ]:
ballots_activity_copeland_2c={Ballot(ranking=[{"J"},{"K"},{"L"},{"M"}],weight=13),Ballot(ranking=[{"K"},{"L"},{"M"},{"J"}],weight=12),
            Ballot(ranking=[{"L"},{"J"},{"K"},{"M"}],weight=11),Ballot(ranking=[{"M"},{"J"},{"K"},{"L"}],weight=3),Ballot(ranking=[{"M"},{"K"},{"L"},{"J"}],weight=3),Ballot(ranking=[{"M"},{"L"},{"J"},{"K"}],weight=3)}
#ballots_activity_copeland_2c={Ballot(ranking=[{"J"},{"K"},{"L"}],weight=13),Ballot(ranking=[{"K"},{"L"},{"J"}],weight=12),Ballot(ranking=[{"L"},{"J"},{"K"}],weight=11)}
#candidates_activity_copeland_2c={"J","K","L"}
candidates_activity_copeland_2c = {"J","K","L","M"}
profile_activity_copeland_2c = PreferenceProfile(ballots=ballots_activity_copeland_2c, candidates=candidates_activity_copeland_2c)
profile_df_head(profile_activity_copeland_2c, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">Run the cell below to see the results of the election and any Condorcet candidate.

In [ ]:
copeland_scores_2c_cond = copeland_score(profile_activity_copeland_2c)
print()
print("Copeland scores:", copeland_scores_2c_cond)
print()
copeland_results(profile_activity_copeland_2c)
print()
pwc_graph_copeland_2c = PairwiseComparisonGraph(profile_activity_copeland_2c)

# condorcet winner
if pwc_graph_copeland_2c.has_condorcet_winner() == True:
    print("The Condorcet candidate is:", pwc_graph_copeland_2c.get_condorcet_winner())
else:
    print("There is no Condorcet candidate.")

<font size="4">Run the cell below to see the pairwise comparison graph illustrating the results of the election.

In [ ]:
pwc_graph_copeland_2c = PairwiseComparisonGraph(profile_activity_copeland_2c)
pwc_graph_copeland_2c.draw()

<font size="4"> From example 6.2 we can see two things:
- Copeland elections do not always have a Condorcet candidate.

- We got a tie. (This is possible in other voting systems, but it is much more likely in Copeland due to the point system.)

## Why is having a Condorcet candidate important?

<font size="4">Social choice theorists consider the existence of a Condorcet candidate important because it demonstrates that the voters prefer that candidate when paired against any other candidate. This is strong support from the electorate. 

This brings us to our next fairness criteria.

# Section 7: Fairness criterion: Condorcet

<font size="4"> _If an election has a Condorcet candidate, then that candidate should win the election._

In other words, if a candidate wins all of its pairwise comparisons, then that candidate should win the election.

When using the Copeland method a Condorcet candidate will always win because that candidate will have a perfect score. The question is: What will happen to a Condorcet candidate when using another voting method?

The Condorcet criterion is sometimes referred to as the head-to-head criterion.

## Example 7.1

<font size="4">Does plurality voting always satisfy the Condorcet criteria? 

In order to that the Condorcet criterion has been violated we must show that 
- the election had a Condorcet candidate, and
- the Condorcet candidate did not win the election.

Otherwise we say that the election did not violate the Condorcet criteria.

Let's look at a few plurailty elections.

<font size="4">a) Run the cell below to view the preference table for a plurality election involving candidates D, E and F.

In [ ]:
candidates_hh_1=["D","E","F"]

ballots_plurality_hh_1=[Ballot(ranking=[{"D"},{"E"},{"F"}],weight=10),Ballot(ranking=[{"D"},{"F"},{"E"}],weight=11),Ballot(ranking=[{"E"},{"D"},{"F"}],weight=6),
                        Ballot(ranking=[{"E"},{"F"},{"D"}],weight=7),Ballot(ranking=[{"F"},{"D"},{"E"}],weight=9),Ballot(ranking=[{"F"},{"E"},{"D"}],weight=8)]

profile_plurality_hh_1 = PreferenceProfile(ballots=ballots_plurality_hh_1, candidates=candidates_hh_1)

profile_df_head(profile_plurality_hh_1, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">Run the cell below to see the results of this plurality election.

In [ ]:
plurality_results(profile_plurality_hh_1)

<font size="4">Run the cell below to see if this election violated the Condorcet criteria.

In [ ]:
condorcet_criteria_test_plurality(profile_plurality_hh_1)

<font size="4">b) Run the cell below to view the preference table for a plurality election involving candidates A, B and C. This is the preference table from example 2.1.

In [ ]:
ballots_plurality_hh=[Ballot(ranking=[{"A"},{"C"},{"B"}]),Ballot(ranking=[{"A"},{"B"},{"C"}]),Ballot(ranking=[{"B"},{"C"},{"A"}]),Ballot(ranking=[{"C"},{"A"},{"B"}])
,Ballot(ranking=[{"C"},{"A"},{"B"}]),Ballot(ranking=[{"A"},{"C"},{"B"}]),Ballot(ranking=[{"C"},{"B"},{"A"}]),Ballot(ranking=[{"B"},{"C"},{"A"}])
,Ballot(ranking=[{"C"},{"A"},{"B"}]),Ballot(ranking=[{"B"},{"A"},{"C"}]),Ballot(ranking=[{"A"},{"C"},{"B"}]),Ballot(ranking=[{"B"},{"C"},{"A"}])
,Ballot(ranking=[{"A"},{"B"},{"C"}]),Ballot(ranking=[{"C"},{"B"},{"A"}]),Ballot(ranking=[{"C"},{"B"},{"A"}]),Ballot(ranking=[{"B"},{"A"},{"C"}])
,Ballot(ranking=[{"C"},{"B"},{"A"}]),Ballot(ranking=[{"A"},{"B"},{"C"}]),Ballot(ranking=[{"A"},{"B"},{"C"}]),Ballot(ranking=[{"B"},{"C"},{"A"}])
,Ballot(ranking=[{"A"},{"B"},{"C"}])]
candidates=["A","B","C"]
profile_plurality_hh = PreferenceProfile(ballots=ballots, candidates=candidates)

candidates=["A","B","C"]

grouped_profile_plurality_hh = profile_plurality_hh.group_ballots()

profile_df_head(grouped_profile_plurality_hh, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">Run the cell below to see the results of this plurality election.

In [ ]:
plurality_results(grouped_profile_plurality_hh)

<font size="4">Run the cell below to see if this election violated the Condorcet criteria.

In [ ]:
condorcet_criteria_test_plurality(grouped_profile_plurality_hh)

<font size="4">c) Run the cell below to view the preference table for a plurality election involving candidates G, H and I.

In [ ]:
candidates_hh_1c=["G","H","I"]

ballots_plurality_hh_1c=[Ballot(ranking=[{"G"},{"H"},{"I"}],weight=10),Ballot(ranking=[{"G"},{"I"},{"H"}],weight=11),Ballot(ranking=[{"H"},{"G"},{"I"}],weight=5),
                        Ballot(ranking=[{"H"},{"I"},{"G"}],weight=14),Ballot(ranking=[{"I"},{"G"},{"H"}],weight=5),Ballot(ranking=[{"I"},{"H"},{"G"}],weight=14)]

profile_plurality_hh_1c = PreferenceProfile(ballots=ballots_plurality_hh_1c, candidates=candidates_hh_1c)

profile_df_head(profile_plurality_hh_1c, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">Run the cell below to see the results of this plurality election.

In [ ]:
plurality_results(profile_plurality_hh_1c)

<font size="4">Run the cell below to see if this election violated the Condorcet criteria.

In [ ]:
condorcet_criteria_test_plurality(profile_plurality_hh_1c)

## Activity 7.2

<font size="4">Does instant-runoff voting always satisfy the Condorcet criteria?

You will examine three real-world instant-runoff elections and determine whether each election violated the Condorcet criteria.

Remember, in order to that the Condorcet criterion has been violated we must show that 
- the election had a Condorcet candidate, and
- the Condorcet candidate did not win the election.

Otherwise we say that the election did not violate the Condorcet criteria.

<font size="4">a) The 2022 Oakland, California District 4 school director instant-runoff election had three candidates: Mike Hutchinson (MH), Nick Resnick (NR), and Pecolia Manigo (PM). Run the cell below to view the preference table for election.

In [ ]:
candidates = ["MH", "NR", "PM", "WI"]
# MIKE HUTCHINSON; NICK RESNICK; PECOLIA MANIGO
oakland_school_board_D4_2022_ballots = []

with open("CVR files/Oakland_2022_School_Board_D4.csv", newline="", encoding="utf-8-sig") as ballotfile:
    for row in csv.reader(ballotfile):
        if not row or row[0].strip().lower() == "ballotid":
            continue

        ranking = [
            {candidate.strip()}
            for candidate in row[1:]
            if candidate.strip()
        ]

        oakland_school_board_D4_2022_ballots.append(
            Ballot(ranking=ranking, weight=1)
        )

profile_oakland_school_board_D4_2022 = PreferenceProfile(
    ballots = oakland_school_board_D4_2022_ballots,
    candidates=candidates
)

profile_oakland_school_board_D4_2022_clean_temp_1=remove_cand_rank_profile("WI", profile_oakland_school_board_D4_2022, remove_empty_ballots = False)
profile_oakland_school_board_D4_2022_clean_temp_2 = remove_repeat_cands_rank_profile(profile_oakland_school_board_D4_2022_clean_temp_1)
profile_oakland_school_board_D4_2022_clean_temp_3 = condense_rank_profile(profile_oakland_school_board_D4_2022_clean_temp_2)

profile_oakland_school_board_D4_2022_clean = profile_oakland_school_board_D4_2022_clean_temp_3.group_ballots()

print(profile_oakland_school_board_D4_2022_clean)
for i in range(0, len(profile_oakland_school_board_D4_2022_clean.ballots), 15):
    display(profile_df_head(profile_oakland_school_board_D4_2022_clean, i + 15, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").drop(index=range(0, i), inplace=False).transpose())

<font size="4">Run the cell below to see the results of this instant-runoff election.

In [ ]:
IRV_results_undervotes(profile_oakland_school_board_D4_2022_clean)

<font size="4">Run the cell below to see if this election had a Condorcet candidate.

In [ ]:
pwc_graph_oakland_school_board_D4_2022_clean = PairwiseComparisonGraph(profile_oakland_school_board_D4_2022_clean)

# condorcet winner
if pwc_graph_oakland_school_board_D4_2022_clean.has_condorcet_winner() == True:
    print("The Condorcet candidate is:", pwc_graph_oakland_school_board_D4_2022_clean.get_condorcet_winner())
else:
    print("There is no Condorcet candidate.")

<font size="4">Did the election violate the Condorcet criteria? Run the cell to answer.

In [ ]:
condorcet_criteria_question_IRV(profile_oakland_school_board_D4_2022_clean)

<font size="4">b) The 2022 Alaska U.S. Senate instant-runoff election had four candidates: Buzz A. Kelley (BK), Kelly C. Tshibaka, Lisa Murkowski (LM), and Patricia R. Chesbro (PC). Run the cell below to view the preference table for the election.

In [ ]:
candidates = ["BK", "KT", "LM", "PC", "WI"]
# Kelley, Buzz A.; Tshibaka, Kelly C.; Murkowski, Lisa; Chesbro, Patricia R.
alaska_senate_2022_ballots = []

with open("CVR files/Alaska_2022_US_Senator.csv", newline="", encoding="utf-8-sig") as ballotfile:
    for row in csv.reader(ballotfile):
        if not row or row[0].strip().lower() == "ballotid":
            continue

        ranking = [
            {candidate.strip()}
            for candidate in row[1:]
            if candidate.strip()
        ]

        alaska_senate_2022_ballots.append(
            Ballot(ranking=ranking, weight=1)
        )

profile_alaska_senate_2022 = PreferenceProfile(
    ballots=alaska_senate_2022_ballots,
    candidates=candidates
)

profile_alaska_senate_2022_clean_temp_1=remove_cand_rank_profile("WI", profile_alaska_senate_2022, remove_empty_ballots = False)
profile_alaska_senate_2022_clean_temp_2 = remove_repeat_cands_rank_profile(profile_alaska_senate_2022_clean_temp_1)
profile_alaska_senate_2022_clean_temp_3 = condense_rank_profile(profile_alaska_senate_2022_clean_temp_2)

profile_alaska_senate_2022_clean = profile_alaska_senate_2022_clean_temp_3.group_ballots()

print(profile_alaska_senate_2022_clean)
for i in range(0, len(profile_alaska_senate_2022_clean.ballots), 15):
    display(profile_df_head(profile_alaska_senate_2022_clean, i + 15, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").drop(index=range(0, i), inplace=False).transpose())

<font size="4">Run the cell below to view the results for the election.

In [ ]:
IRV_results_complete(profile_alaska_senate_2022_clean)
print()

<font size="4">Run the cell below to see if this election had a Condorcet candidate.

In [ ]:
pwc_graph_alaska_senate_2022_clean = PairwiseComparisonGraph(profile_alaska_senate_2022_clean)

# condorcet winner
if pwc_graph_alaska_senate_2022_clean.has_condorcet_winner() == True:
    print("The Condorcet candidate is:", pwc_graph_alaska_senate_2022_clean.get_condorcet_winner())
else:
    print("There is no Condorcet candidate.")

<font size="4">Did the election violate the Condorcet criteria? Run the cell to answer.

In [ ]:
condorcet_criteria_question_IRV(profile_alaska_senate_2022_clean)

<font size="4">c) The 2022 Alaska U.S. Special House instant run-off election had three candidates: Mary S. Peltola, Nick Begich, and Sarah Palin. Run the cell below to view the preference table for the election.

In [ ]:
candidates = ["MP", "NB", "SP", "WI"]
# Peltola, Mary S.; Begich, Nick; Palin, Sarah
alaska_house_spec_2022_ballots = []

with open("CVR files/Alaska_2022_US_House_Spec.csv", newline="", encoding="utf-8-sig") as ballotfile:
    for row in csv.reader(ballotfile):
        if not row or row[0].strip().lower() == "ballotid":
            continue

        ranking = [
            {candidate.strip()}
            for candidate in row[1:]
            if candidate.strip()
        ]

        alaska_house_spec_2022_ballots.append(
            Ballot(ranking=ranking, weight=1)
        )

profile_alaska_house_spec_2022 = PreferenceProfile(
    ballots=alaska_house_spec_2022_ballots,
    candidates=candidates
)

profile_alaska_house_spec_2022_clean_temp_1=remove_cand_rank_profile("WI", profile_alaska_house_spec_2022, remove_empty_ballots = False)
profile_alaska_house_spec_2022_clean_temp_2 = remove_repeat_cands_rank_profile(profile_alaska_house_spec_2022_clean_temp_1)
profile_alaska_house_spec_2022_clean_temp_3 = condense_rank_profile(profile_alaska_house_spec_2022_clean_temp_2)

profile_alaska_house_spec_2022_clean = profile_alaska_house_spec_2022_clean_temp_3.group_ballots()

print(profile_alaska_house_spec_2022_clean)
for i in range(0, len(profile_alaska_house_spec_2022_clean.ballots), 15):
    display(profile_df_head(profile_alaska_house_spec_2022_clean, i + 15, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").drop(index=range(0, i), inplace=False).transpose())

<font size="4">Run the cell below to view the results for the election.

In [ ]:
IRV_results_undervotes(profile_alaska_house_spec_2022_clean)
print()

<font size="4">Run the cell below to see if this election had a Condorcet candidate.

In [ ]:
pwc_graph_alaska_house_spec_2022_clean = PairwiseComparisonGraph(profile_alaska_house_spec_2022_clean)

# condorcet winner
if pwc_graph_alaska_house_spec_2022_clean.has_condorcet_winner() == True:
    print("The Condorcet candidate is:", pwc_graph_alaska_house_spec_2022_clean.get_condorcet_winner())
else:
    print("There is no Condorcet candidate.")

<font size="4">Did the election violate the Condorcet criteria? Run the cell to answer.

In [ ]:
condorcet_criteria_question_IRV(profile_alaska_house_spec_2022_clean)

## Fairness criteria and real-world elections

<font size="4">From activity 7.2 we can see that both plurality and instant-runoff systems may violate the Condorcet criteria. How often does this occur in real-world elections?

In the case of plurality elections, it is diffifult to determine because plurality elections in the real-world do not use ranked ballots.

Since instant-runoff elections do use ranked ballots in the real-world social choice researchers have looked at how often this occurs.

Social choice researchers examined 185 U.S. instant-runoff elections. Of these, 183 had a Condorcet candidate. Only the Burlington, Vermont 2009 mayoral election and the Alaska 2022 congressional election violated the Condorcet criteria. In the other 181 elections the Condorcet candidate won. [Graham-Squire, McCune 2022]



# Section 8: Borda Count Elections

<font size="4">Borda count is a point based method. Candidates receive points based upon the voters' rankings. It is primarily used by organizations and associations. Like other ranked voting methods it allows voters to express preferences for a number of candidates. Additionally, the method gives some weight to all of the voters' preferences.

## Steps to run a Borda count election

<font size="3"> The value k is equal to the number of candidates.

    i. Each voter ranks the candidates.
    
    ii. For each nth choice vote a candidate receives k-n+1 points.*

    iii. The candidate with most total points is declared the winner.

<font size="4">* For a 1st choice vote the candidate receives k-1+1=k points. For a 2nd choice vote the candidate receives k-2+1=k-1 points. And so on.

## Example 8.1

<font size="4">Suppose there is an organization called Americans for Electoral Reform. Its members need to decide where to have their next convention. The following cities are options: Albuquerque (A), Baltimore (B), Houston (H), and Sacramento (S). The organization wants to find a city that is generally agreeable to all the members. They decide that the Borda count method will be the best way to do this. Let's compute the results of the election.

a) The cities have been ranked by the voters (step (i)). Run the cell below to view the preference table for the election.

In [ ]:
ballots_convention={Ballot(ranking=[{"A"},{"H"},{"S"},{"B"}],weight=6),Ballot(ranking=[{"H"},{"A"},{"B"},{"S"}],weight=9),
             Ballot(ranking=[{"S"},{"A"},{"H"},{"B"}],weight=7),Ballot(ranking=[{"B"},{"H"},{"A"},{"S"}],weight=8),
             Ballot(ranking=[{"S"},{"B"},{"A"},{"H"}],weight=4),Ballot(ranking=[{"H"},{"B"},{"S"},{"A"}],weight=12),
             Ballot(ranking=[{"S"},{"A"},{"B"},{"H"}],weight=7),Ballot(ranking=[{"A"},{"S"},{"B"},{"H"}],weight=6)}
candidates_convention={"A","B","H","S"}
profile_convention = PreferenceProfile(ballots=ballots_convention, candidates=candidates_convention)
profile_df_head(profile_convention, 10, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">b) Step (ii) compute each cities total points. There are four cities, thus k=4. First choice votes (n=1) are worth k-n+1=4-1+1=4 points each. Second choice votes (n=2) are worth k-n+1=4-2+1=3 points each. Third choice votes (n=3) are worth k-n+1=4-3+1=2 points each. Fourth choice votes (n=4) are worth k-n+1=4-4+1=1 point each.

Run the cell below to view the computation for each city.

In [ ]:
borda_candidate_score(profile_convention, 'A')
print()

borda_candidate_score(profile_convention, 'B')
print()

borda_candidate_score(profile_convention, 'H')
print()

borda_candidate_score(profile_convention, 'S')

<font size="4">b) Step (iii) declare city with the most points the winner. Compare total points in part (b), then run cell belwo to check your answer.

In [ ]:
locals={'A':'Albuqurque', 'B':'Baltimore', 'H': 'Houston', 'S': 'Sacramento'}
city=borda_results(profile_convention)[0]

print(f'The convention will be held in {locals[city]}.')

## Activity 8.2

<font size="4">In this activity you will compute the results of a Borda count election. (The ballots are randomized so you can repeat this activity for more practice.)

<font size="4">(a) Step (i) is for voters to rank the candidates. Run the cell below to generate the preference table.

In [ ]:
slate_to_candidates = {"Alpha": ["A", "B"], "Xenon": ["X", "Y"]}

# note that we include candidates with 0 support,
# and that our preference intervals will automatically rescale to sum to 1

preference_mapping = {
    "Alpha": {
        "Alpha": PreferenceInterval({"A": 0.7, "B": 0.3}),
        "Xenon": PreferenceInterval({"X": 0.4, "Y": 0.6}),
    },
    "Xenon": {
        "Alpha": PreferenceInterval({"A": 0.4, "B": 0.6}),
        "Xenon": PreferenceInterval({"X": 0.7, "Y": 0.3}),
    },
}


bloc_proportions = {"Alpha": 0.5, "Xenon": 0.5}

# assume that each bloc is 90% cohesive
# we'll discuss exactly what that means later
cohesion_mapping = {
    "Alpha": {"Alpha": 0.9, "Xenon": 0.1},
    "Xenon": {"Xenon": 0.9, "Alpha": 0.1},
}

config = bg.BlocSlateConfig(
    n_voters=301,
    preference_mapping=preference_mapping,
    bloc_proportions=bloc_proportions,
    slate_to_candidates=slate_to_candidates,
    cohesion_mapping=cohesion_mapping,
)

# by using the by_bloc generator we can see which ballots came from which blocs of voters
profile_dict = bg.name_pl_profiles_by_bloc_generator(config)
# print("The ballots from Alpha voters\n", profile_df_head(profile_dict["Alpha"], 10))

# print("The ballots from Xenon voters\n", profile_df_head(profile_dict["Xenon"], 10))

# to create the aggregate profile, we just sum the profiles in the dictionary
agg_profile_borda_temp_1 = profile_dict["Alpha"] + profile_dict["Xenon"]
agg_profile_borda = agg_profile_borda_temp_1.group_ballots()

# display(profile_df_head(agg_profile_borda, 24, sort_by_weight=False, percents=False, totals=True, n_decimals=3).drop(columns="Voter Set").drop(index='Total').transpose())
for i in range(0, len(agg_profile_borda.ballots), 12):
    display(profile_df_head(agg_profile_borda, i + 12, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").drop(index=range(0, i), inplace=False).transpose())

<font size="4">(b) Complete step (ii) by computing the total points for each candidate. Then run the cell below to check your answers.

In [ ]:
for cand in borda_scores(agg_profile_borda):
    while True:
        try:
            x = float(input(f"How many points did candidate {cand} receive?"))
            break
        except:
            print("Invalid input")
    if x == borda_scores(agg_profile_borda)[cand]:
        print(f"Your total for candidate {cand} is correct.")
    else:
        print(f"Your total for candidate {cand} is incorrect.")
print("If any of your answers were incorrect, correct answers and repeat part (b). Otherwise, continue to part (c).")

<font size="4">(c) Complete step (iii) by determining which candidate won the election or the set of candidates that tied. Run the cell below to enter your answer.

In [ ]:
if len(borda_results(agg_profile_borda)) == 1:
    while True:
        w = input("Did the election result in a tie? (y/n):")
        if w == 'y' or w == 'n':
            break
        else:
            print("Invalid input")
    if w == "n":
        print("Correct. The election did not result in a tie.")
        x=input("Enter the candidate that won the election.")
        if x ==  borda_results(agg_profile_borda)[0]:
            print(f"Correct. {borda_results(agg_profile_borda)[0]} won the election.")
        else:
            print("Incorrect. Run cell again and give it another try.")
    else:
        print("Incorrect. The election did result in a tie. Run cell again and give it another try.")
if len(borda_results(agg_profile_borda)) > 1:
    while True:
            t = input("Did the election result in a tie? (y/n):")
            if t == 'y' or t == 'n':
                break
            else:
                print("Invalid input")
    if t == "y":
        print("Correct. The election did result in a tie.")
        while True:
            try:
                n = int(input("How many candidates were involved in the tie? Enter an integer (no decimals)."))
                break
            except:
                print("Invalid input")
        list = []
        for i in range(0,int(n)):
            x=input("Enter a candidate that was involved in the tie that you have not already entered.")
            list.append(x)
        if list == borda_results(agg_profile_borda):
            print(f"Correct. The candidates {list} were tied.")
    else:
        print("Incorrect. Run cell again and give it another try.")

# Section 9: Fairness Criteria: Majority

<font size="4"> _If a candidate receives a majority of the first choice votes, then that candidate should win the election._

<font size="4">Remember, a majority is more than half. 

We will refer to any candidate that receives a majority of the first choice votes as the _majority candidate_.

## Majority candidates are always Condorcet candidates

<font size="4">Suppose a candidate A receives a majority of first-choice votes (over 50%). Since candidate A has a majority of first-choice votes, when compared to any other candidate B, candidate B will have less than 50% of the vote. This will result in A winning when compared pairwise versus any other candidate. Thus A will be a Condorcet candidate.

Note that the converse is not necessarilly true. Condorcet candidates are not necessarilly majority candidates.

## Activity 9.1

<font size="4">In this activity you will try to determine which voting systems will always satisfy the Majority criterion. 

To decide if a voting system always satisfies the Majority criterion, first imagine that a candidate did receive a majority of first choice votes. If under that voting system the candidate is guaranteed to win the election, you can say the voting system will always satisfy the Majority criterion.

If you think the voting system may sometimes violate the Majority criterion, you should try to come up with an example to support your conclusion. 



<font size="4">a) Does the plurality method always satisfy the Majority criterion? Run the cell below to enter your answer.

In [ ]:
while True:
    x = input('Does the plurality method always satisfy the Majority criterion? (y/n):')
    if x == 'y' or x == 'n':
        break
    else:
        print("Invalid input")
if x == 'y':
    print("Correct. The plurality method alsways satisfies the Majority criterion.")
else:
    print("Incorrect. It is not possible for the plurality method to violate the Majority criterion in any circumstance.")
print()
print('Explanation: Suppose a candidate A receives a majority of first-choice votes (over 50%).')
print('The most any other candidate can receive is less than 50%.')
print('Thus A has the most first-choice votes and wins the election.')


<font size="4">b) Does the instant-runoff method always satisfy the Majority criterion? Run the cell below to enter your answer.

In [ ]:
while True:
    x = input('Does the instant-runoff method always satisfy the Majority criterion? (y/n):')
    if x == 'y' or x == 'n':
        break
    else:
        print("Invalid input")
if x == 'y':
    print("Correct. The instant-runoff method alsways satisfies the Majority criterion.")
else:
    print("Incorrect. It is not possible for the instant-runoff method to violate the Majority criterion in any circumstance.")
print()
print('Explanation: Suppose a candidate A receives a majority of first-choice votes (over 50%).')
print('In round 1, if a candidate receives a majority of the first-choice votes. That candidate wins and the election is over.')
print('Thus A wins the election.')

<font size="4">c) Does the Copeland pairwise comparison method always satisfy the Majority criterion? Run the cell below to enter your answer.

In [ ]:
while True:
    x = input('Does the Copeland pairwise comparison method always satisfy the Majority criterion? (y/n):')
    if x == 'y' or x == 'n':
        break
    else:
        print("Invalid input")
if x == 'y':
    print("Correct. The Copeland pairwise comparison method alsways satisfies the Majority criterion.")
else:
    print("Incorrect. It is not possible for the Copeland pairwise comparison method to violate the Majority criterion in any circumstance.")
print()
print('Explanation: Suppose a candidate A receives a majority of first-choice votes (over 50%).')
print('Since candidate A is the majority candidate, A is also the Condorcet candidate.')
print('Thus A wins when compared pairwise to any other candidate. Hence A wins the election.')

<font size="4">d) Does the Borda count method always satisfy the Majority criterion? Run the cell below to enter your answer.

In [ ]:
while True:
    x = input('Does the Borda count method always satisfy the Majority criterion? (y/n):')
    if x == 'y' or x == 'n':
        break
    else:
        print("Invalid input")
if x == 'n':
    print("Correct. It is possible for the Borda count method to violate the Majority criterion under some circumstances.")
else:
    print("Incorrect. The Borda count method does not always satisfy the Majority criterion.")
print()
print("In the following activity you will create a hypothetical Borda count election that violates the Majority criterion.")

## Activity 9.2

<font size="4">In this activity you will show that Borda count method does not always satisfy the Majority criterion.

<font size="4">a) Suppose there is a Borda count election with three candidates X, Y and Z. There are six possible permutations of the rankings: X>Y>Z, X>Z>Y, Y>X>Z, Y>Z>X, Z>X>Y and Z>Y>X. 

You will need to choose the number of each type of ballot such that
- one candidate has a majority of the first choice votes, and
- a different candidate wins the election.

Run the cell below to enter your numbers and view the preference table for the election.

In [ ]:
while True:
    try:
        a=float(input("Enter the number of ballots with ranking X>Y>Z"))
        break
    except:
        print("Invalid input")

while True:
    try:
        b=float(input("Enter the number of ballots with ranking X>Z>Y"))
        break
    except:
        print("Invalid input")

while True:
    try:
        c=float(input("Enter the number of ballots with ranking Y>X>Z"))
        break
    except:
        print("Invalid input")

while True:
    try:
        d=float(input("Enter the number of ballots with ranking Y>Z>X"))
        break
    except:
        print("Invalid input")

while True:
    try:
        e=float(input("Enter the number of ballots with ranking Z>X>Y"))
        break
    except:
        print("Invalid input")

while True:
    try:
        f=float(input("Enter the number of ballots with ranking Z>Y>X"))
        break
    except:
        print("Invalid input")

ballots_borda_activity=[Ballot(ranking=[{"X"},{"Y"},{"Z"}],weight=a),Ballot(ranking=[{"X"},{"Z"},{"Y"}],weight=b),Ballot(ranking=[{"Y"},{"X"},{"Z"}],weight=c),Ballot(ranking=[{"Y"},{"Z"},{"X"}],weight=d)
,Ballot(ranking=[{"Z"},{"X"},{"Y"}],weight=e),Ballot(ranking=[{"Z"},{"Y"},{"X"}],weight=f)]
candidates_borda_activity=["X","Y","Z"]
profile_borda_activity = PreferenceProfile(ballots=ballots_borda_activity, candidates=candidates_borda_activity)

if majority_test(profile_borda_activity) == None:
    print("Invalid input. No candidate has a first-choice majority.")
    print("Choose inputs so that a candidate has a first-choice vote majority.")
else:
    print(f"{majority_test(profile_borda_activity)} has a majority of first choice votes.")
    display(profile_df_head(profile_borda_activity, 6, sort_by_weight=False, percents=True, totals=True, n_decimals=3).drop(columns={"Voter Set",'Percent'},index='Total').transpose())


<font size="4">b) Run the cell below to see how many points each candidate received.

In [ ]:
borda_candidate_score(profile_borda_activity, 'X')
print()

borda_candidate_score(profile_borda_activity, 'Y')
print()

borda_candidate_score(profile_borda_activity, 'Z')
print()

<font size="4">c) Which candidate won the Borda count election? Run the cell below to enter your answer.

In [ ]:
if len(borda_results(profile_borda_activity)) == 1:
    while True:
        w = input("Did the election result in a tie? (y/n):")
        if w == 'y' or w == 'n':
            break
        else:
            print("Invalid input")
    if w == "n":
        print("Correct. The election did not result in a tie.")
        x=input("Enter the candidate that won the election.")
        if x ==  borda_results(profile_borda_activity)[0]:
            print(f"Correct. {borda_results(profile_borda_activity)[0]} won the election.")
        else:
            print("Incorrect. Run cell again and give it another try.")
    else:
        print("Incorrect. The election did result in a tie. Run cell again and give it another try.")
if len(borda_results(agg_profile_borda)) > 1:
    while True:
            t = input("Did the election result in a tie? (y/n):")
            if t == 'y' or t == 'n':
                break
            else:
                print("Invalid input")
    if t == "y":
        print("Correct. The election did result in a tie.")
        while True:
            try:
                n = int(input("How many candidates were involved in the tie? Enter an integer (no decimals)."))
                break
            except:
                print("Invalid input")
        list = []
        for i in range(0,int(n)):
            x=input("Enter a candidate that was involved in the tie that you have not already entered.")
            list.append(x)
        if list == borda_results(profile_borda_activity):
            print(f"Correct. The candidates {list} were tied.")
    else:
        print("Incorrect. Run cell again and give it another try.")

<font size="4">d) Run the cell below to see if your Borda count election violated the Majority criterion.

In [ ]:
if borda_results(profile_borda_activity)[0] == majority_test(profile_borda_activity):
    print(f"The majority candidate {majority_test(profile_borda_activity)} also won the Borda count election.") 
    print("The Majority criterion was not violated. Try activity again.")
else:
    print(f"The majority candidate was {majority_test(profile_borda_activity)}")
    print(f"Yet, the winner of the Borda count election was {borda_results(profile_borda_activity)[0]}")
    print("The election violated the Majority criterion.")
    print("You have shown that the Borda count method does not always satisfy the Majority criterion.")

# Section 10: Approval Voting

<font size="4"> Instant-runoff, Copeland pairwise comparison and Borda count are all ordinal methods which require these types of elections to use a ranked ballot. Plurality is a non-ordinal method since only a voters first-choice is counted. Usually plurality elctions are run using a single mark ballot. Approval voting is another non-ordinal method. This method uses approval ballots that allow voters to vote for any number of candidates that are on the ballot.

## Steps to run an Approval election

<font size="3"> 

    i. Each voter approves of any number of candidates on the ballot. 
    
    ii. Each approval is a vote for that candidate. Find the total votes for each candidate.
    
    iii. The candidate with the most votes wins.

<font size="4">Approval voting is simple. It is designed to decrease vote-splitting between similar candidates (thus minimizing the spoiler effect.) Supporters also say that it incentivizes candidates to form broad coalitions and encourages voters to vote more sincerely. [Ballotpedia, 2026] Approval voting is used both by organizations and political elections. For political elections it is not as common as instant-runoff. 

St. Louis, Missouri uses a multi-winner version of approval voting for its local primary elections. Fargo, North Dakota used approval voting for local elections from 2022 until the state banned instant-runoff and approval voting in the state in 2025. The Utah state Forward Party used approval voting to select a replacement for a state senator who resigned. [Ballotpedia, 2026]

## Example 10.1

<font size="4">Suppose there is an approval election with candidates A, B, C, D, and E. Let's compute the results of the election.

<font size="4">a) For step (i) each voter must approve of candidates on the ballot. Run the cell below to view the preference table for the election.  

In [ ]:
approval_ballots_1=[
    Ballot(scores={"A":1,"B":1,"C":1,"D":1},weight=6),
    Ballot(scores={"A":1,"D":1,"E":1},weight=5),
    Ballot(scores={"A":1,"B":1,"E":1},weight=4),
    Ballot(scores={"A":1},weight=10),
    Ballot(scores={"B":1,"E":1,"D":1},weight=6),
    Ballot(scores={"B":1},weight=5),
    Ballot(scores={"C":1,"B":1},weight=7),
    Ballot(scores={"C":1,"D":1},weight=8),
    Ballot(scores={"C":1},weight=9),
    Ballot(scores={"C":1,"E":1,"D":1},weight=3),
    Ballot(scores={"C":1,"D":1},weight=5),
    Ballot(scores={"D":1,"E":1},weight=9),
    Ballot(scores={"D":1},weight=4)]
candidates_approval_1={"A","B","C","D","E"}
profile_approval_1 = PreferenceProfile(ballots=approval_ballots_1, candidates=candidates_approval_1)
profile_df_head(profile_approval_1, 15, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">This is a bit different from a ranked ballot. The candidates are listed in the first column. If a ballot shows a 1.0 in that candidates row that candidate was approved by the voter. Otherwise it says "NaN". For instance, ballots with index 7 show that 8 voters approved of candidates D and C and did not approve of the others. 

<font size="4">b) In step (ii) we will find the total votes for each candidate. For each candidate, look for the columns marked with a 1.0 in the preference table. Add up the wieghts in those columns. That is the candidates vote total. Give it a try, then run the cell below to see the details.

In [ ]:
approval_details(profile_approval_1)

<font size="4">c) Step (iii) determine the winner of the election. Run the cell below to view the results.

In [ ]:
approval_result_statement(profile_approval_1)

# Section 11: Approval Voting and the Fairness Criteria

<font size="4">Approval voting is not a ranked based election method, but we can still examine whether approval voting violates fairness criteria. We do need to make a couple of assumptions. 
- First, that each voter has a ranking of the candidates even though the ranking cannot be expressed on an approval ballot. 
- Second, if a voter approves a candidate, they approve all candidates they ranked higher.

## Example 11.1

<font size="4">Suppose there is an approval election for candidates X, Y and Z. Based upon the assumptions above, let's examine the hypothetical preferences of 9 voters and some possible resulting approval ballots.

<font size="4">a) Run the cell below to view the ranked preferences of the 9 voters.

In [ ]:
candidates_ranked={"X","Y","Z"}

ballots_ranked=[Ballot(ranking=[{"X"},{"Y"},{"Z"}],weight=1),
                Ballot(ranking=[{"X"},{"Z"},{"Y"}],weight=1),
                Ballot(ranking=[{"Y"},{"X"},{"Z"}],weight=1),
                #Ballot(ranking=[{"Y"},{"Z"},{"X"}],weight=1),
                Ballot(ranking=[{"Z"},{"X"},{"Y"}],weight=1),
                Ballot(ranking=[{"Z"},{"Y"},{"X"}],weight=1),
                Ballot(ranking=[{"X"},{"Y"},{"Z"}],weight=1),
                Ballot(ranking=[{"X"},{"Z"},{"Y"}],weight=1),
                #Ballot(ranking=[{"Y"},{"X"},{"Z"}],weight=1),
                #Ballot(ranking=[{"Y"},{"Z"},{"X"}],weight=1),
                Ballot(ranking=[{"Z"},{"X"},{"Y"}],weight=1),
                Ballot(ranking=[{"X"},{"Z"},{"Y"}],weight=1)
]

profile_ranked = PreferenceProfile(ballots=ballots_ranked, candidates=candidates_ranked)

#print(profile_ranked.df.to_string())

profile_df_head(profile_ranked, 12, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns={"Voter Set",'Weight'}).transpose()

<font size="4">b) Based upon our assumptions about voters' behavior a voter can approve up to three candidates but will do so in order of their ranked preferences. Run the cell below to see how this might work out for the 9 voters from part (a).

In [ ]:
candidates_ranked={"X","Y","Z"}

ballots_scored=[Ballot(scores = {"X":1,"Y":1},weight=1),
                Ballot(scores = {"X":1,"Z":1},weight=1),
                Ballot(scores = {"Y":1},weight=1),
                #Ballot(ranking=[{"Y"},{"Z"},{"X"}],weight=1),
                Ballot(scores = {"Z":1},weight=1),
                Ballot(scores = {"Z":1, "Y":1},weight=1),
                Ballot(scores = {"X":1},weight=1),
                Ballot(scores = {"X":1,"Z":1},weight=1),
                #Ballot(ranking=[{"Y"},{"X"},{"Z"}],weight=1),
                #Ballot(scores = {"Y":1, "Z":1},weight=1),
                Ballot(scores = {"X":1,"Z":1},weight=1),
                Ballot(scores = {"X":1, "Z":1},weight=1)
]

profile_scored = PreferenceProfile(ballots=ballots_scored, candidates=candidates_ranked)

#print(profile_scored.df.to_string())

profile_df_head(profile_scored, 12, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns={"Voter Set","Weight"}).transpose()

<font size="4">Notice that the voter (with ballot index) 0 and the voter (with ballot index) 5 both ranked the candidates X>Y>Z. But they voted differently when they filled out the ballots for the approval election, voter 0 approved of X and Y, while voter 5 approved of X only. Voter 0 maximized the chance that their least preferred candidate would lose, while voter 5 maximized the chance their favorite candidate would win. Approval voting does not eliminate strategic considerations for voters.

## Activity 11.2

<font size="4">Suppose the Summer Movie Series shows free movies in the park weekly. The 170 attendees get to vote on what movie will be shown the following week. Organizers use approval voting. The movies on the ballot are:
- The Lego Movie (L)
- Moana (M)
- The SpongeBob Movie: The Searsch for Squarepants (S)
- Toy Story 3 (T)

<font size="4">a) Run the cell below to view the preferences of the attendees.

In [ ]:
movies = {"L","M","S","T"}

movie_ballots_ranked = [Ballot(ranking=[{"T"},{"M"},{"S"},{"L"}],weight=100),
            Ballot(ranking=[{"M"},{"L"},{"T"},{"S"}],weight=50),
            Ballot(ranking=[{"S"},{"L"},{"M"},{"T"}],weight=20),
]

profile_movies_ranked = PreferenceProfile(ballots=movie_ballots_ranked, candidates=movies)

profile_df_head(profile_movies_ranked, 12, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">b) Voters can approve of any number of movies so long as they do so in order of preference. In this case we will assume all voters approve of their top two movies. The voters with ballot index zero will approve of Toy Story 3 (T) and Moana (M). The voters with ballot index 1 will approve of Moana (M) and The Lego Movie (L). The voters with ballot index 2 will approve of The SpongeBob Movie (S) and The Lego Movie (L). Run the cell below to view the resulting preference table for the approval election.

In [ ]:
profile_movies_scored = convert_rank_profile_to_score_profile_via_score_vector(profile_movies_ranked,(1,1,0,0))

profile_df_head(profile_movies_scored, 12, sort_by_weight=False, percents=False, totals=False, n_decimals=3).drop(columns="Voter Set").transpose()

<font size="4">c) Compute the winner of the approval election. Run the cell below to enter your answer.

In [ ]:
while True:
        answer = input("Which movie won the election? (L, M, S or T):")
        if answer in movies:
            break
        else:
            print("Invalid input")
r = Approval(profile = profile_movies_scored).get_elected()[0]
approval_winner, = r
if answer == approval_winner:
     print(f"Correct. {approval_winner} is the winning movie.")
else:
    print("Incorrect. Re-run cell and try again.")


<font size="4">To view details of election run cell below.

In [ ]:
approval_details(profile_movies_scored)

approval_result_statement(profile_movies_scored)

<font size="4">d) Is their a majority movie? (i.e. Does any movie have a majority of the first choice preferences?) To determine this you will need to examine the ranked preferences from part (a). Run the cell below to enter your answer.

In [ ]:
while True:
    m = input("Is there a majority movie? (y/n):")
    if m == 'y' or m == 'n':
        break
    else:
        print("Invalid input")
if majority_test(profile_movies_ranked) != None:
    if m == 'y':
        while True:
            h = input("Which had the majority of first choice preferences? (L, M, S, or T):")
            if h in movies:
                break
            else:
                print("Invalid input")
        if h == majority_test(profile_movies_ranked):
            print(f"Correct. {majority_test(profile_movies_ranked)} is the majority movie.")
        else:
            print("Incorrect. That is not the majority movie. Run the cell again and give it another try.")
    else:
        print("Incorrect. There is a majority movie. Run the cell again and give it another try.")
else:
    if m == 'y':
        print("Incorrect. There is no majority movie. Run the cell again and give it another try.")
    else:
        print("Correct. There is no majority movie.")

    

<font size="4">e) Did the election violate the Majority criterion? Run the cell below to enter your answer.

In [ ]:
majority_question_approval(profile_movies_ranked, profile_movies_scored)

<font size="4">f) This example also shows that the approval election violated one of the other fairness criteria. Which one was it?

    A. Condorcet

    B. Independence of Irrelevant Alternatives

    C. Monotonicity

Run the cell below to enter your answer.

In [ ]:
while True:
    d = input('Which fairness criteria did the approval election violate? (A, B or C):')
    if d == 'A' or d == 'B' or d == 'C':
        break
    else:
        print("Invalid input")
if d =='A':
    print(f"Correct. Since {majority_test(profile_movies_ranked)} was the majority candidate, {majority_test(profile_movies_ranked)} is also the Condorcet candidate.")
    print(f"Yet {majority_test(profile_movies_ranked)} did not win the election.")
else:
    print('Incorrect. Run the cell again and give it another try.')

# Section 12: Is there a perfect voting system?

<font size="4">If there are only two candidates, then the anwer is yes. Unfortunately, there are problems if there are three or more candidates. First, is it always possible for a voting system to translate the input from the electorate into a single election winner output? Secondly, are there voting systems that satisfy all the fairness criteria?

## Example 12.1

<font size="4">In this example we will explore a case known as "Condorcet's voting paradox" identified by the French mathematician and philosopher Marquis de Condorcet [Condorcet 1785].

Suppose an electorate consists of three voters who have ranked three candidates A, B and C.

<font size="4">a) Run the cell below to view the voters preferences.

In [ ]:
ballots_cp=[Ballot(ranking=[{"A"},{"C"},{"B"}]),Ballot(ranking=[{"B"},{"A"},{"C"}]),Ballot(ranking=[{"C"},{"B"},{"A"}])]
candidates=["A","B","C"]
profile_cp = PreferenceProfile(ballots=ballots_cp, candidates=candidates)

profile_df_head(profile_cp, 7, sort_by_weight=False, percents=True, totals=True, n_decimals=3).drop(columns={"Voter Set",'Weight','Percent'},index='Total').transpose()

<font size="4">b) Now let's try to make an argument for which candidate is the most deserving to win. No candiate has a majority or even a plurality of first choice votes. In fact, all candidates are tied with one first choice vote (the same is true for second and thid choice votes). We will need to dig a bit deeper. Let's look at the pairwise comparisons to start. If there is a Condorcet candidate, that would be a strong argument in favor of that candidate. Go ahead and list the possible pairings. Run the cell below to check your answer.

In [ ]:
pwc_graph_cp = PairwiseComparisonGraph(profile_cp)
pwc_graph_cp.pairwise_dict.keys()

for i in range (0, len(pwc_graph_cp.pairwise_dict.keys())):
    print(list(pwc_graph_cp.pairwise_dict.keys())[i])

<font size="4">c) Now determine which candidate wins each pairing. Run the cell below for an explanation.

In [ ]:
pwc_graph_cp.pairwise_dict

for (cand1, cand2), (cand1_votes, cand2_votes) in pwc_graph_cp.pairwise_dict.items():
    if cand1_votes > cand2_votes:
        print(f"{cand1} receives {cand1_votes} votes and {cand2} receives {cand2_votes} votes.  {cand1} defeats {cand2}.")
    elif cand2_votes > cand1_votes:
        print(f"{cand2} receives {cand2_votes} votes and {cand1} receives {cand1_votes} votes.  {cand2} defeats {cand1}.")
    else:
        print(f"{cand2} receives {cand2_votes} votes and {cand1} receives {cand1_votes} votes.  {cand1} and {cand2} tie.")

print()
print("There is no Condorcet candidate. Something strange has happened.")
print("Since B defeats A and A defeats C we might expect B to defeat C, but this is not the case.")
print("It turns out that C defeats B, creating a cycle of preferences.")
print()
print("In the pairwise comparison graph below, you can see the cycle of preferences.")
print("This is what is known as 'Condorcet's paradox'.")

pwc_graph_cp.draw()

<font size="4">As mentioned plurality method won't produce a winner. Part (c) shows that Copeland pairwise comparison will not produce a winner. You should check instant-runoff and Borda count, but as you will quickly see they also will not produce a winner. If any candidate is somehow declared the winner, two-thirds of voters prefer another candidate!

Condorcet's paradox shows us that there is at least one set of inputs from the electorate for which it appears there is no reasonable way to select a winner.

## How 'fair' can a voting system be?

<font size="4">In this notebook we have examined various voting systems under these fairness criteria:
- Independence of Irrelevant Alternatives
- Monotonicity
- Condorcet
- Majority

We might wonder is there a voting system that always satisfies all four of these criteria simultaneously?

For elections with ranked ballots and three or more candidates the answer is no. Kenneth Arrow proved this in 1951 [Arrow, 1951 [1963]]. His result is now referred to as Arrow's Impossibility theorem. Arrow went on to win the Nobel Prize in economics for his work in 1972.

The following statement of the theorem is the one that is typical of most undergraduate texts on the subject.

## Arrow's Impossibilty Theorem

<font size="4">There is no ranked voting system with three or more candidates that satisfies:
- Independence of Irrelevant Alternatives
- Monotonicity
- Condorcet
- Majority

[Johnson, Mowry 2016]

Note that this applies to all ranked voting systems that already exist and any that are yet to be invented!

Arrow's original statement of the theorem is more technical. Arrow's proof utilized five fairness criteria: Unrestricted Domain, Social Ordering, Weak Pareto, Nondictatorship and Independence of Irrelevant Alternatives [Morreau 2025]. However, failure to satisfy one of Arrow's fairness criteria also guarantees the voting system will fail to satisfy one of the four fairness criteria stated above.

## What should be done?

<font size="4">Even though it seems that the search for a perfect voting system is futile, there may still be a voting system that is better than the others. Some voting systems do incentivize strategic (insincere) voting more than others [Eggers, Nowicki (forthcoming)]. We have seen throughout this notebook that different voting systems fail different fairness criteria. And we have also seen that even when a voting system has the potential to violate a fairness criteria that in the real world this may happen infrequently.

Electoral reformers need to be informed about the relative advantages and disadvantages of different voting systems. Then promote those that are shown theoretically and empirically to have the desired qualities. Reformers can do so by following the results of social choice research.

# References

<font size="4">Arrow, K.J. 1951 [1963]. Social Choice and Individual Values, New York: Wiley. 2nd ed.

<font size="4">Ballotpedia article (website). 2026. "Approval Voting". https://ballotpedia.org/Approval_voting

<font size="4">Data and Democracy Lab (website). 2026. "Votekit". https://data-democracy.org/posts/votekit

<font size="4">Eggers, Andrew and Tobias Nowicki. Forthcoming. “Susceptibility to Strategic Voting: A Comparison of Plurality and Instant-Runoff Elections”. _Journal of Politics_.

<font size="4"> Felsenthal, Dan S. and Tideman, Nicolaus. 2014. "Interacting double monotonicity failure with direction of impact under five voting methods". _Mathematical Social Sciences_.

<font size="4">Graham-Squire, Adam and McCune, David. 2022. "An Examination of Ranked-Choice Voting in the United States, 2004–2022". _Representation_.

<font size="4">Johnson, David B. and Mowry, Thomas A. 2016. "6.1 Voting Systems". _Mathematics : A Practicasl Odyssey_.

<font size="4">Lavelle, Harrison and Thomas, Armin. 2023. "Examining Ross Perot’s Impact on the 1992 Presidential Election Results". _Split Ticket_. https://split-ticket.org/2023/04/01/examining-ross-perots-impact-on-the-1992-presidential-election/

<font size="4">Morreau, Michael. 2025 "Arrow’s Theorem", The Stanford Encyclopedia of Philosophy (Winter 2025 edition), https://plato.stanford.edu/archives/win2025/entries/arrows-theorem/

<font size="4">Nicolas de Condorcet (Marquis de Condorcet). 1785. "Essai sur l'application de l'analyse à la probabilité des décisions rendues à la pluralité des voix"

<font size="4">Olesinski, Ethan. 2025. "PoliSciPy: A Python library for U.S. Electoral College visualization (Version 0.1.0) [Software]". _GitHub_. https://github.com/poliscipy/poliscipy.

<font size="4">Stephanopoulos, Nicholas O. 2024. "Finding Condorcet". _Washington and Lee Law Review_.

<font size="4">Schlesinger, Joseph A. and Schlesinger, Mildred S. 2006 "Maurice Duverger and the Study of Political Parties". _French Politics_

<font size="4">Wikipedia article (website). 2026. "1992 United States presidential election". https://en.wikipedia.org/wiki/1992_United_States_presidential_election#Democratic_Party_nomination


# Appendix

## Activity: What if Ross Perot didn't run in 1992?

<font size="4">In 1992 Texas billionarire H. Ross Perot contested the U.S. presidential election and received the largest vote share (approx. 19%) for any third-party candidate since Theodore Roosevelt in 1912. Perot did not win any electoral votes, but may have played the role of spoiler in many states swinging those electoral votes into the winner Bill Clinton's column.

In this activity you can explore what might have happened had Perot not run. Was Ross Perot a spoiler? Did the 1992 election violate the Independence of Irrelevant Alternatives criterion?

<font size="4">a) There were four presidential candidates on the ballot in all 50 states and the District of Columbia in 1992.
- Bill Clinton (BC) (Democrat)
- George H.W. Bush (GB) (Republican and incumbent president)
- H. Ross Perot (RP) (independent)
- Andre Marrou (AM) (Libertarian)

Every state (except Oklahoma and West Virginia) had various other small party candidates (O).

Run the cell below to view the electoral college map and a summary of the results of the election [Wikipedia 2026]. EV refers to electoral college votes. The vote totals for other small party candidates are combined.

In [ ]:
source_file = "CVR files/1992_US_Presidential_Election.csv"
output_file = "CVR files/1992_US_Presidential_Election_with_Perot.csv"
output_columns = [0, 1, 2, 4, 5, 7, 8,10, 12, 14]

with open(source_file, newline="", encoding="latin-1") as electionfile, \
     open(output_file, "w", newline="", encoding="utf-8") as withperotfile:
    reader = csv.reader(electionfile)
    writer = csv.writer(withperotfile)

    header = next(reader)
    writer.writerow([header[i].strip() for i in output_columns])

    total_ev_clinton = 0
    total_ev_bush = 0
    winning_party = {}

    for row in reader:
        if not row or not row[0].strip() or row[0].strip().casefold() == "state":
            continue

        clinton = int(int(row[2].replace(",", "")))
        bush = int(int(row[5].replace(",", "")))
        
        if row[0].strip().casefold() != "totals:" and clinton > bush:
            ev_clinton = int(row[1])
            ev_bush = 0
            total_ev_clinton += ev_clinton
            total_ev_bush += ev_bush
            winning_party[row[0].strip()] = "Democrat"
        elif row[0].strip().casefold() != "totals:" and bush > clinton:
            ev_clinton = 0
            ev_bush = int(row[1])
            total_ev_clinton += ev_clinton
            total_ev_bush += ev_bush
            winning_party[row[0].strip()] = "Republican"

        if row[0].strip().casefold() != "totals:":
            new_row = [
                row[0].strip(), row[1], clinton, ev_clinton,
                bush, ev_bush, row[8],row[10], row[12], clinton+bush+int(row[10].replace(",", ""))+int(row[8].replace(",", ""))+int(row[12].replace(",", ""))
            ]
            writer.writerow(new_row)
        elif row[0].strip().casefold() == "totals:":
            total_row = [
                row[0].strip(), row[1], clinton, total_ev_clinton,
                bush, total_ev_bush, row[8], row[10], row[12], clinton+bush+int(row[10].replace(",", ""))+int(row[8].replace(",", ""))+int(row[12].replace(",", ""))
            ]
            writer.writerow(total_row)

if total_ev_clinton > total_ev_bush:
    print("Clinton wins the election!")
elif total_ev_bush > total_ev_clinton:
    print("Bush wins the election!")
else:
    print("The election is a tie!")

# Load U.S. electoral college geospatial data
gdf = load_shapefile(year="1992")

# Merge the winning party data with the GeoDataFrame
gdf['winning_party'] = gdf['STUSPS'].map(winning_party).fillna('No Data')

plot_electoral_map(
    gdf,
    column="winning_party",
    year="1992",
    title="1992 U.S. Presidential Election",
    legend=True,
    vote_bar=True
)
voteset = get_votes_wp()
#print(voteset.columns.tolist())
display(voteset.head(52))

<font size="4">b) If Perot had not run his votes would presumably would have have gone to other candidates, the second choice preference of those voters. Since plurality elections don't actually use ranked ballots we can't know for sure what those preferences were. One analysis of the election concludes that 51% of Perot's votes would have gone to Clinton, 42% to Bush and 7% would have voted for other candidates or not voted [Lavelle, Thomas 2023].

Suppose Perot is removed from the ballot. Run the cell below to redistribute his votes to the other candidates and view the new summary table. Make sure your entrees total 100%.

In [ ]:
while True:
    x = input("What percent of Perot's votes should be redistributed to Clinton? (Enter a number between 0 and 100): ")
    y = input("What percent of Perot's votes should be redistributed to Bush? (Enter a number between 0 and 100): ")
    z = input("What percent of Perot's votes should be redistributed to Marrou? (Enter a number between 0 and 100): ")
    w = input("What percent of Perot's votes should be redistributed to other candidates? (Enter a number between 0 and 100): ")
    if float(x) + float(y) + float(z) + float(w) == 100:
        break
    else:
        print("Invalid input. Please enter values that sum to 100.")

redistribute = [float(x) / 100, float(y) / 100, float(z) / 100, float(w) / 100]

source_file = "CVR files/1992_US_Presidential_Election.csv"
output_file = "CVR files/1992_US_Presidential_Election_No_Perot.csv"
output_columns = [0, 1, 2, 4, 5, 7, 10, 12, 14]

with open(source_file, newline="", encoding="latin-1") as electionfile, \
     open(output_file, "w", newline="", encoding="utf-8") as noperotfile:
    reader = csv.reader(electionfile)
    writer = csv.writer(noperotfile)

    header = next(reader)
    writer.writerow([header[i].strip() for i in output_columns])

    total_ev_clinton = 0
    total_ev_bush = 0
    winning_party_np = {}

    for row in reader:
        if not row or not row[0].strip() or row[0].strip().casefold() == "state":
            continue

        clinton = int(int(row[2].replace(",", "")) + int(row[8].replace(",", "")) * redistribute[0])
        bush = int(int(row[5].replace(",", "")) + int(row[8].replace(",", "")) * redistribute[1])
        marrou = int(int(row[10].replace(",", "")) + int(row[8].replace(",", "")) * redistribute[2])
        other = int(int(row[12].replace(",", "")) + int(row[8].replace(",", "")) * redistribute[3])
        total_votes = clinton+bush+marrou+other

        if row[0].strip().casefold() != "totals:" and clinton > bush:
            ev_clinton = int(row[1])
            ev_bush = 0
            total_ev_clinton += ev_clinton
            total_ev_bush += ev_bush
            winning_party_np[row[0].strip()] = "Democrat"
        elif row[0].strip().casefold() != "totals:" and bush > clinton:
            ev_clinton = 0
            ev_bush = int(row[1])
            total_ev_clinton += ev_clinton
            total_ev_bush += ev_bush
            winning_party_np[row[0].strip()] = "Republican"

        if row[0].strip().casefold() != "totals:":
            new_row = [
                row[0].strip(), row[1], clinton, ev_clinton,
                bush, ev_bush, marrou, other, total_votes
            ]
            writer.writerow(new_row)
        elif row[0].strip().casefold() == "totals:":
            total_row = [
                row[0].strip(), row[1], clinton, total_ev_clinton,
                bush, total_ev_bush, marrou, other, total_votes
            ]
            writer.writerow(total_row)

print(f'When {redistribute[0]*100}% of Perot\'s votes are redistributed to Clinton, {redistribute[1]*100}% to Bush, {redistribute[2]*100}% to Marrou, and {redistribute[3]*100}% to other candidates:')
print()

if total_ev_clinton > total_ev_bush:
    print("Clinton wins the election!")
elif total_ev_bush > total_ev_clinton:
    print("Bush wins the election!")
else:
    print("The election is a tie!")

voteset_np = get_votes_np()
#print(voteset_np.columns.tolist())
voteset_np.head(52)

# Load U.S. electoral college geospatial data
gdf = load_shapefile(year="1992")

# Merge the winning party data with the GeoDataFrame
gdf['winning_party'] = gdf['STUSPS'].map(winning_party_np).fillna('No Data')

plot_electoral_map(
    gdf,
    column="winning_party",
    year="1992",
    title="Hypothetical 1992 U.S. Presidential Election without Perot",
    legend=True,
    vote_bar=True
)
voteset = get_votes_np()
#print(voteset.columns.tolist())
display(voteset.head(52))

<font size="4">c) Repeat part (b) several times to see which redistributions of Perot's votes result in a Clinton win and which result in a Bush win. Do you think it is likely that Perot a spoiler?

In [1]:



import nbformat

path = "Voting_Systems_alpha.ipynb"
notebook = nbformat.read(path, as_version=4)

for cell in notebook.cells:
    if cell.cell_type == "code":
        cell.metadata["editable"] = False
        cell.metadata.setdefault("jupyter", {})["source_hidden"] = True

nbformat.write(notebook, path)